In [ ]:
# Runs an end-to-end test of the complete system on 5 videos.

# Test système End-to-End — Child Hazard Monitoring

Ce notebook réalise **un test système complet de bout en bout** sur 5 vidéos représentatives :

`vidéo brute → extraction 16 frames → YOLO26m enfant → VideoMAE activité → YOLOE contexte → distances enfant–objet → JSON factuel → Mistral-7B-Instruct-v0.3 few-shot → hazard/no_hazard`

## Configuration finale gelée

La configuration de décision utilisée ici est **celle sélectionnée dans la Partie 4** :

- modèle : `mistralai/Mistral-7B-Instruct-v0.3`
- mode : **few-shot**
- 4 démonstrations issues **uniquement de la calibration** : 2 `hazard` + 2 `no_hazard`
- sortie imposée : JSON `{"hazard": "...", "reason": "..."}`
- aucune adaptation QLoRA n'est utilisée dans le système final.

## Fichiers/datasets Kaggle nécessaires

Ajouter comme Inputs Kaggle :

1. `Part4JSON` — les vidéos calibration/test ;
2. `Yolo26Good` — `best (3).pt` ;
3. `videomae-final-rhad-result` — `checkpoint-best.pth` ;
4. **les deux fichiers gelés de la Partie 4** :
   - `calibration_inputs.json`
   - `ground_truth_calibration_annotated.csv`

> Important : il n'existe pas de « modèle Partie 4 gelé » à télécharger. Le modèle final est le **Mistral de base**, chargé depuis Hugging Face, avec un prompt few-shot gelé. Les deux petits fichiers ci-dessus suffisent à reconstruire exactement les 4 exemples de calibration.

Les 5 cas système choisis couvrent : danger direct, situation sûre, activité contextuelle sûre, activité contextuelle dangereuse et objet tranchant.


In [ ]:
# ============================================================
# CELLULE 1 — ENVIRONNEMENT + DÉTECTION DES DATASETS KAGGLE
# ============================================================

%pip install --no-cache-dir -q -U ultralytics

from pathlib import Path
import json, math, os, re, shutil, subprocess, sys, time, zipfile
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO, YOLOE

print("Ultralytics :", ultralytics.__version__)

INPUT_ROOT = Path("/kaggle/input")
DATASETS_ROOT = INPUT_ROOT / "datasets"
assert DATASETS_ROOT.exists(), f"Dossier absent : {DATASETS_ROOT}"

def find_dataset(keyword):
    keyword = keyword.lower()
    candidates = [
        p for p in DATASETS_ROOT.rglob("*")
        if p.is_dir()
        and keyword in p.name.lower()
        and any(x.is_file() for x in p.rglob("*"))
    ]
    print(f"\nCandidats pour '{keyword}' :")
    for p in candidates:
        print(" -", p)
    assert candidates, f"Dataset contenant '{keyword}' introuvable."
    return sorted(candidates, key=lambda p: len(p.parts), reverse=True)[0]

PART4_DATASET = find_dataset("part4json")
VIDEOMAE_DATASET = find_dataset("videomae-final-rhad-result")
YOLO_DATASET = find_dataset("yolo26good")

NUM_CONTEXT_FRAMES = 16
OBJECT_CONFIDENCE_THRESHOLD = 0.14
NORMAL_CLASS_INDEX = 7
CLASS_NAMES = [
    "Jumping","child_running","climbing","collesion",
    "electrical_appliances","falling","head_stuck","normal",
    "objects_falling","pulling_curtains","scribbling","sharp_objects",
]
EXPECTED_NUMBER_OF_VIDEOS = 5
EXPECTED_NUMBER_OF_FRAMES = EXPECTED_NUMBER_OF_VIDEOS * NUM_CONTEXT_FRAMES

PROJECT_DIR = Path("/kaggle/working/E2E_final_test")
FRAMES_DIR = PROJECT_DIR / "extracted_frames"
VISUALIZATIONS_DIR = PROJECT_DIR / "visualizations"
JSON_OUTPUTS_DIR = PROJECT_DIR / "json_outputs"
ANNOTATIONS_DIR = PROJECT_DIR / "annotations"
MANIFESTS_DIR = PROJECT_DIR / "manifests"
MODULE_OUTPUTS_DIR = PROJECT_DIR / "module_outputs"

for d in [PROJECT_DIR,FRAMES_DIR,VISUALIZATIONS_DIR,JSON_OUTPUTS_DIR,
          ANNOTATIONS_DIR,MANIFESTS_DIR,MODULE_OUTPUTS_DIR]:
    d.mkdir(parents=True,exist_ok=True)

print("Part4    :",PART4_DATASET)
print("VideoMAE :",VIDEOMAE_DATASET)
print("YOLO     :",YOLO_DATASET)
print("\nCELLULE 1 TERMINÉE")


In [ ]:
# ============================================================
# CELLULE 2 — SÉLECTION DE 5 VIDÉOS TEST + EXTRACTION 16 FRAMES
# ============================================================

from pathlib import Path
import re
import shutil
import time
import zipfile

PART4_WORK_DIR = Path("/kaggle/working/e2e_part4_dataset")

if PART4_WORK_DIR.exists():
    shutil.rmtree(PART4_WORK_DIR)

PART4_WORK_DIR.mkdir(parents=True, exist_ok=True)

zip_candidates = sorted(PART4_DATASET.rglob("*.zip"))

if zip_candidates:
    print("ZIP utilisé :", zip_candidates[0])
    with zipfile.ZipFile(zip_candidates[0], "r") as archive:
        archive.extractall(PART4_WORK_DIR)
    SEARCH_ROOT = PART4_WORK_DIR
else:
    print("Aucun ZIP : utilisation directe du dataset Kaggle.")
    SEARCH_ROOT = PART4_DATASET

test_candidates = [p for p in SEARCH_ROOT.rglob("test") if p.is_dir()]
assert len(test_candidates) == 1, test_candidates
TEST_ROOT = test_candidates[0]

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}

def list_videos(root):
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )

test_videos = list_videos(TEST_ROOT)
assert len(test_videos) == 24, f"24 vidéos test attendues, obtenu : {len(test_videos)}"

def sanitize_name(text):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(text)).strip("_")

# Construire le manifeste complet du split test afin de retrouver
# exactement les mêmes video_id que dans la Partie 4.
records = []
counters = {}

for video_path in test_videos:
    class_name = video_path.parent.name
    assert class_name in CLASS_NAMES, f"Classe inconnue : {class_name}"

    counters[class_name] = counters.get(class_name, 0) + 1
    candidate_number = counters[class_name]

    capture = cv2.VideoCapture(str(video_path))
    assert capture.isOpened(), video_path

    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()

    if fps <= 0:
        fps = 30.0

    duration_seconds = float(total_frames) / float(fps) if total_frames > 0 else 0.0

    video_id = (
        f"test__{sanitize_name(class_name)}__"
        f"{candidate_number:02d}__{sanitize_name(video_path.stem)}"
    )

    records.append({
        "video_id": video_id,
        "split": "test",
        "class_name": class_name,
        "candidate_number": candidate_number,
        "filename": video_path.name,
        "video_path": str(video_path),
        "duration_seconds": duration_seconds,
        "total_frames": total_frames,
        "fps": fps,
        "width": width,
        "height": height,
    })

all_test_videos_df = pd.DataFrame(records).sort_values(
    ["class_name", "candidate_number"]
).reset_index(drop=True)

# 5 scénarios représentatifs
E2E_EXPECTED = {
    "test__falling__01__1_11": "hazard",
    "test__normal__01__1_16": "no_hazard",
    "test__child_running__01__1_11": "no_hazard",
    "test__pulling_curtains__02__1_31": "hazard",
    "test__sharp_objects__01__1_61": "hazard",
}

missing_ids = sorted(set(E2E_EXPECTED) - set(all_test_videos_df["video_id"].astype(str)))
assert not missing_ids, (
    "Les video_id attendus n'ont pas été retrouvés : "
    + ", ".join(missing_ids)
)

selected_videos_df = (
    all_test_videos_df[
        all_test_videos_df["video_id"].isin(E2E_EXPECTED)
    ]
    .copy()
    .reset_index(drop=True)
)

selected_videos_df["expected_hazard"] = (
    selected_videos_df["video_id"].map(E2E_EXPECTED)
)

EXPECTED_NUMBER_OF_VIDEOS = len(selected_videos_df)
EXPECTED_NUMBER_OF_FRAMES = EXPECTED_NUMBER_OF_VIDEOS * NUM_CONTEXT_FRAMES

assert EXPECTED_NUMBER_OF_VIDEOS == 5
assert selected_videos_df["expected_hazard"].notna().all()

selected_videos_df.to_csv(
    MANIFESTS_DIR / "e2e_selected_videos.csv",
    index=False,
)

display(
    selected_videos_df[
        ["video_id", "class_name", "expected_hazard", "video_path"]
    ]
)

# ------------------------------------------------------------
# Extraction uniforme de 16 frames / vidéo
# ------------------------------------------------------------

if FRAMES_DIR.exists():
    shutil.rmtree(FRAMES_DIR)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

def extract_frames(video_row):
    video_path = Path(video_row["video_path"])
    video_id = str(video_row["video_id"])

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Impossible d'ouvrir {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps <= 0:
        fps = 30.0

    assert total_frames > 0, video_path

    indices = np.linspace(
        0, total_frames - 1, NUM_CONTEXT_FRAMES, dtype=int
    )

    output_dir = FRAMES_DIR / video_id
    output_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    for sample_index, frame_index in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()

        if not ok:
            raise RuntimeError(
                f"Frame illisible : {video_path} / {frame_index}"
            )

        timestamp_seconds = float(frame_index) / float(fps)

        frame_path = output_dir / (
            f"sample_{sample_index:02d}_frame_{frame_index:05d}.jpg"
        )

        assert cv2.imwrite(str(frame_path), frame)

        extracted.append({
            "video_id": video_id,
            "split": "test",
            "class_name": str(video_row["class_name"]),
            "candidate_number": int(video_row["candidate_number"]),
            "video_filename": video_path.name,
            "video_path": str(video_path),
            "sample_index": int(sample_index),
            "frame_index": int(frame_index),
            "timestamp_seconds": float(timestamp_seconds),
            "fps": float(fps),
            "width": int(width),
            "height": int(height),
            "frame_path": str(frame_path),
        })

    cap.release()
    return extracted

start = time.perf_counter()
all_frames = []

for _, row in selected_videos_df.iterrows():
    all_frames.extend(extract_frames(row))

FRAME_EXTRACTION_TOTAL_SECONDS = time.perf_counter() - start
extracted_frames_df = pd.DataFrame(all_frames)

assert len(extracted_frames_df) == EXPECTED_NUMBER_OF_FRAMES
assert extracted_frames_df.groupby("video_id").size().eq(NUM_CONTEXT_FRAMES).all()

extracted_frames_df.to_csv(
    MANIFESTS_DIR / "e2e_extracted_frames.csv",
    index=False,
)

print("\n" + "=" * 80)
print("E2E — SÉLECTION ET EXTRACTION TERMINÉES")
print("=" * 80)
print("Vidéos :", EXPECTED_NUMBER_OF_VIDEOS)
print("Frames :", len(extracted_frames_df))
print("Temps  :", round(FRAME_EXTRACTION_TOTAL_SECONDS, 2), "s")
print("\nCELLULE 2 TERMINÉE")


In [ ]:
# ==============================================================================
# DÉTECTION DE L'ENFANT AVEC YOLO26M
# ==============================================================================

# ============================================================
# CELLULE 5A — DÉTECTION DE L'ENFANT AVEC YOLO26M
# SUR LES FRAMES E2E
# ============================================================

from pathlib import Path
import shutil
import time
import gc

import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from ultralytics import YOLO


# ------------------------------------------------------------
# 1. Retrouver le checkpoint du dataset Yolo26Good
# ------------------------------------------------------------

possible_matches = sorted(
    path
    for path in YOLO_DATASET.rglob("best*.pt")
    if path.is_file()
)

print("Checkpoints YOLO trouvés :")
for checkpoint_path in possible_matches:
    print(" -", checkpoint_path)

assert len(possible_matches) >= 1, (
    "Aucun checkpoint best*.pt trouvé dans Yolo26Good."
)

# Si plusieurs fichiers existent, préférer best (3).pt puis best.pt.
preferred = [
    p for p in possible_matches
    if p.name == "best (3).pt"
]
if not preferred:
    preferred = [
        p for p in possible_matches
        if p.name == "best.pt"
    ]

uploaded_checkpoint_path = (
    preferred[0] if preferred else possible_matches[0]
)

print("\nCheckpoint sélectionné :", uploaded_checkpoint_path)
print(
    "Taille :",
    f"{uploaded_checkpoint_path.stat().st_size / 1024**2:.2f} Mo",
)


# ------------------------------------------------------------
# 2. Copier sous un nom simple
# ------------------------------------------------------------

CHILD_MODEL_PATH = Path(
    "/kaggle/working/"
    "yolo26m_child_best.pt"
)

shutil.copy2(
    uploaded_checkpoint_path,
    CHILD_MODEL_PATH,
)

assert CHILD_MODEL_PATH.exists()


# ------------------------------------------------------------
# 3. Charger YOLO26m
# ------------------------------------------------------------

CHILD_CONFIDENCE_THRESHOLD = 0.25
CHILD_IMAGE_SIZE = 640

child_model = YOLO(
    str(CHILD_MODEL_PATH)
)

print(
    "\nModèle chargé :",
    CHILD_MODEL_PATH,
)

print(
    "Classes du modèle :",
    child_model.names,
)


# ------------------------------------------------------------
# 4. Préparer les dossiers de sortie
# ------------------------------------------------------------

CHILD_DETECTIONS_DIR = (
    PROJECT_DIR
    / "yolo26m_child_detections"
)

CHILD_ANNOTATED_FRAMES_DIR = (
    CHILD_DETECTIONS_DIR
    / "annotated_frames"
)

CHILD_MONTAGES_DIR = (
    CHILD_DETECTIONS_DIR
    / "montages"
)

for directory in [
    CHILD_DETECTIONS_DIR,
    CHILD_ANNOTATED_FRAMES_DIR,
    CHILD_MONTAGES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 5. Vérifier les 768 frames
# ------------------------------------------------------------

ordered_frames_df = (
    extracted_frames_df
    .sort_values(
        [
            "class_name",
            "video_id",
            "sample_index",
        ]
    )
    .reset_index(drop=True)
)

assert len(ordered_frames_df) == EXPECTED_NUMBER_OF_FRAMES, (
    f"{EXPECTED_NUMBER_OF_FRAMES} frames attendues, "
    f"mais {len(ordered_frames_df)} trouvées."
)


# ------------------------------------------------------------
# 6. Détecter l'enfant image par image
# ------------------------------------------------------------

child_detection_records = []
child_frame_summary_records = []
child_primary_inference_seconds = 0.0


for frame_position, (_, frame_row) in enumerate(

    tqdm(
        ordered_frames_df.iterrows(),
        total=len(ordered_frames_df),
        desc="Détection YOLO26m",
    ),

    start=1,
):

    video_id = str(
        frame_row["video_id"]
    )

    activity_class = str(
        frame_row["class_name"]
    )

    sample_index = int(
        frame_row["sample_index"]
    )

    frame_index = int(
        frame_row["frame_index"]
    )

    frame_path = Path(
        frame_row["frame_path"]
    )

    assert frame_path.exists(), (
        f"Frame absente : {frame_path}"
    )


    # --------------------------------------------------------
    # Inférence sur une seule frame
    # --------------------------------------------------------

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    child_inference_start = time.perf_counter()

    results = child_model.predict(
        source=str(frame_path),
        conf=CHILD_CONFIDENCE_THRESHOLD,
        imgsz=CHILD_IMAGE_SIZE,
        device=0,
        half=True,
        batch=1,
        verbose=False,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    child_primary_inference_seconds += (
        time.perf_counter() - child_inference_start
    )

    result = results[0]


    # --------------------------------------------------------
    # Sauvegarder l'image annotée
    # --------------------------------------------------------

    video_output_directory = (
        CHILD_ANNOTATED_FRAMES_DIR
        / video_id
    )

    video_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    annotated_frame_path = (
        video_output_directory
        / frame_path.name
    )

    annotated_image = result.plot()

    cv2.imwrite(
        str(annotated_frame_path),
        annotated_image,
    )


    # --------------------------------------------------------
    # Extraire toutes les bounding boxes enfant
    # --------------------------------------------------------

    current_frame_detections = []

    if (
        result.boxes is not None
        and len(result.boxes) > 0
    ):

        for detection_index, box in enumerate(
            result.boxes
        ):

            confidence = float(
                box.conf.item()
            )

            class_id = int(
                box.cls.item()
            )

            class_name_detected = str(
                result.names[class_id]
            )

            x1, y1, x2, y2 = (
                box.xyxy[0]
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

            detection_record = {
                "video_id": video_id,
                "true_activity_class": activity_class,

                "sample_index": sample_index,
                "frame_index": frame_index,

                "timestamp_seconds": float(
                    frame_row[
                        "timestamp_seconds"
                    ]
                ),

                "frame_path": str(
                    frame_path
                ),

                "annotated_frame_path": str(
                    annotated_frame_path
                ),

                "detection_index": int(
                    detection_index
                ),

                "detected_class_id": class_id,
                "detected_class_name": (
                    class_name_detected
                ),

                "confidence": confidence,

                "bbox_x1": float(x1),
                "bbox_y1": float(y1),
                "bbox_x2": float(x2),
                "bbox_y2": float(y2),

                "bbox_width": float(
                    x2 - x1
                ),

                "bbox_height": float(
                    y2 - y1
                ),

                "detector_model": (
                    "YOLO26m child detector"
                ),
            }

            current_frame_detections.append(
                detection_record
            )

            child_detection_records.append(
                detection_record
            )


    # --------------------------------------------------------
    # Garder la détection ayant la meilleure confiance
    # --------------------------------------------------------

    if len(current_frame_detections) > 0:

        best_child_detection = max(
            current_frame_detections,
            key=lambda item: item[
                "confidence"
            ],
        )

        child_detected = True

        best_confidence = (
            best_child_detection[
                "confidence"
            ]
        )

        best_bbox_x1 = (
            best_child_detection[
                "bbox_x1"
            ]
        )

        best_bbox_y1 = (
            best_child_detection[
                "bbox_y1"
            ]
        )

        best_bbox_x2 = (
            best_child_detection[
                "bbox_x2"
            ]
        )

        best_bbox_y2 = (
            best_child_detection[
                "bbox_y2"
            ]
        )

    else:

        child_detected = False

        best_confidence = None

        best_bbox_x1 = None
        best_bbox_y1 = None
        best_bbox_x2 = None
        best_bbox_y2 = None


    child_frame_summary_records.append(
        {
            "video_id": video_id,
            "true_activity_class": activity_class,

            "sample_index": sample_index,
            "frame_index": frame_index,

            "timestamp_seconds": float(
                frame_row[
                    "timestamp_seconds"
                ]
            ),

            "frame_path": str(
                frame_path
            ),

            "annotated_frame_path": str(
                annotated_frame_path
            ),

            "child_detected": bool(
                child_detected
            ),

            "number_of_child_detections": int(
                len(
                    current_frame_detections
                )
            ),

            "child_confidence": (
                best_confidence
            ),

            "child_bbox_x1": (
                best_bbox_x1
            ),

            "child_bbox_y1": (
                best_bbox_y1
            ),

            "child_bbox_x2": (
                best_bbox_x2
            ),

            "child_bbox_y2": (
                best_bbox_y2
            ),
        }
    )


    del results
    del result
    del annotated_image


    if frame_position % 16 == 0:

        gc.collect()

        torch.cuda.empty_cache()

        print(
            f"{frame_position} / {EXPECTED_NUMBER_OF_FRAMES} "
            "frames terminées"
        )


# ------------------------------------------------------------
# 7. Créer les tableaux
# ------------------------------------------------------------

child_detections_df = pd.DataFrame(
    child_detection_records
)

child_frame_summary_df = pd.DataFrame(
    child_frame_summary_records
)

assert len(
    child_frame_summary_df
) == EXPECTED_NUMBER_OF_FRAMES


# ------------------------------------------------------------
# 8. Résumé par vidéo
# ------------------------------------------------------------

child_video_summary_df = (
    child_frame_summary_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
        ]
    )
    .agg(
        frames_analyzed=(
            "frame_path",
            "size",
        ),

        frames_with_child=(
            "child_detected",
            "sum",
        ),

        total_child_detections=(
            "number_of_child_detections",
            "sum",
        ),

        mean_child_confidence=(
            "child_confidence",
            "mean",
        ),

        maximum_child_confidence=(
            "child_confidence",
            "max",
        ),
    )
    .reset_index()
)

child_video_summary_df[
    "child_detection_rate"
] = (
    child_video_summary_df[
        "frames_with_child"
    ]
    /
    child_video_summary_df[
        "frames_analyzed"
    ]
)


# ------------------------------------------------------------
# 9. Sauvegarder les CSV
# ------------------------------------------------------------

CHILD_DETECTIONS_CSV_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_detections.csv"
)

CHILD_FRAME_SUMMARY_CSV_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_frame_summary.csv"
)

CHILD_VIDEO_SUMMARY_CSV_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_video_summary.csv"
)

child_detections_df.to_csv(
    CHILD_DETECTIONS_CSV_PATH,
    index=False,
)

child_frame_summary_df.to_csv(
    CHILD_FRAME_SUMMARY_CSV_PATH,
    index=False,
)

child_video_summary_df.to_csv(
    CHILD_VIDEO_SUMMARY_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Créer quatre images par vidéo sans les afficher
# ------------------------------------------------------------

MONTAGE_SAMPLE_INDICES = [
    0,
    5,
    10,
    15,
]


for video_id in (
    child_frame_summary_df[
        "video_id"
    ].unique()
):

    video_frames = (
        child_frame_summary_df[
            child_frame_summary_df[
                "video_id"
            ] == video_id
        ]
        .sort_values(
            "sample_index"
        )
        .reset_index(drop=True)
    )

    activity_class = str(
        video_frames.loc[
            0,
            "true_activity_class",
        ]
    )

    montage_frames = (
        video_frames[
            video_frames[
                "sample_index"
            ].isin(
                MONTAGE_SAMPLE_INDICES
            )
        ]
        .sort_values(
            "sample_index"
        )
        .reset_index(drop=True)
    )

    figure, axes = plt.subplots(
        2,
        2,
        figsize=(12, 9),
    )

    axes = axes.flatten()

    for axis_index, axis in enumerate(
        axes
    ):

        axis.axis("off")

        if axis_index >= len(
            montage_frames
        ):
            continue

        frame_record = montage_frames.iloc[
            axis_index
        ]

        image_bgr = cv2.imread(
            frame_record[
                "annotated_frame_path"
            ]
        )

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        axis.imshow(
            image_rgb
        )

        confidence = frame_record[
            "child_confidence"
        ]

        confidence_text = (
            f"{confidence:.3f}"
            if pd.notna(confidence)
            else "aucune"
        )

        axis.set_title(
            f"Échantillon "
            f"{frame_record['sample_index']}\n"
            f"Enfant détecté : "
            f"{frame_record['child_detected']} — "
            f"confiance : {confidence_text}",
            fontsize=9,
        )

    figure.suptitle(
        f"{activity_class} — YOLO26m",
        fontsize=14,
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.95,
        ]
    )

    montage_path = (
        CHILD_MONTAGES_DIR
        / f"{video_id}_child_detection.png"
    )

    plt.savefig(
        montage_path,
        dpi=170,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


# ------------------------------------------------------------
# 11. Afficher seulement le résumé
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("YOLO26M — DÉTECTION DE L'ENFANT TERMINÉE")
print("=" * 80)

display(
    child_video_summary_df[
        [
            "true_activity_class",
            "frames_analyzed",
            "frames_with_child",
            "child_detection_rate",
            "mean_child_confidence",
            "maximum_child_confidence",
        ]
    ]
)

print(
    "\nFrames analysées :",
    len(child_frame_summary_df),
)

print(
    "CSV par frame :",
    CHILD_FRAME_SUMMARY_CSV_PATH,
)

print(
    "CSV par vidéo :",
    CHILD_VIDEO_SUMMARY_CSV_PATH,
)

print(
    "Montages :",
    CHILD_MONTAGES_DIR,
)

print("\nCELLULE 5A TERMINÉE")

# ==============================================================================
# SECONDE PASSE SUR LES FRAMES NON DÉTECTÉES
# ==============================================================================

# ============================================================
# CELLULE 5B — SECONDE TENTATIVE SUR LES FRAMES NON DÉTECTÉES
# ============================================================

from pathlib import Path

import cv2
import pandas as pd
import torch

from tqdm.auto import tqdm
from ultralytics import YOLO


# ------------------------------------------------------------
# 1. Configuration du seuil de secours
# ------------------------------------------------------------

FALLBACK_CHILD_CONFIDENCE_THRESHOLD = 0.10
FALLBACK_CHILD_IMAGE_SIZE = 640


# ------------------------------------------------------------
# 2. Recharger le modèle seulement si nécessaire
# ------------------------------------------------------------

if "child_model" not in globals():

    assert CHILD_MODEL_PATH.exists(), (
        f"Modèle introuvable : {CHILD_MODEL_PATH}"
    )

    child_model = YOLO(
        str(CHILD_MODEL_PATH)
    )


# ------------------------------------------------------------
# 3. Copier le tableau principal
# ------------------------------------------------------------

child_frame_summary_fallback_df = (
    child_frame_summary_df
    .copy()
)


child_frame_summary_fallback_df[
    "child_detection_source"
] = "none"

child_frame_summary_fallback_df.loc[
    child_frame_summary_fallback_df[
        "child_detected"
    ] == True,
    "child_detection_source",
] = "primary_threshold_0.25"


# ------------------------------------------------------------
# 4. Identifier uniquement les frames manquées
# ------------------------------------------------------------

missed_frames_df = (
    child_frame_summary_fallback_df[
        child_frame_summary_fallback_df[
            "child_detected"
        ] == False
    ]
    .copy()
)


print("=" * 80)
print("SECONDE TENTATIVE")
print("=" * 80)

print(
    "Frames non détectées lors du premier passage :",
    len(missed_frames_df),
)


# ------------------------------------------------------------
# 5. Dossier des images de secours
# ------------------------------------------------------------

FALLBACK_ANNOTATED_DIR = (
    CHILD_DETECTIONS_DIR
    / "fallback_annotated_frames"
)

FALLBACK_ANNOTATED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 6. Relancer seulement les frames manquées
# ------------------------------------------------------------

fallback_detection_records = []
child_fallback_inference_seconds = 0.0


for dataframe_index, frame_row in tqdm(
    missed_frames_df.iterrows(),
    total=len(missed_frames_df),
    desc="Seconde tentative YOLO26m",
):

    frame_path = Path(
        frame_row["frame_path"]
    )

    assert frame_path.exists(), (
        f"Frame introuvable : {frame_path}"
    )


    # Aucun half=True :
    # cela évite les avertissements affichés précédemment.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    fallback_inference_start = time.perf_counter()

    results = child_model.predict(
        source=str(frame_path),
        conf=FALLBACK_CHILD_CONFIDENCE_THRESHOLD,
        imgsz=FALLBACK_CHILD_IMAGE_SIZE,
        device=0,
        batch=1,
        verbose=False,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    child_fallback_inference_seconds += (
        time.perf_counter() - fallback_inference_start
    )

    result = results[0]


    # --------------------------------------------------------
    # Vérifier si une détection existe
    # --------------------------------------------------------

    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):
        continue


    # --------------------------------------------------------
    # Garder la détection de meilleure confiance
    # --------------------------------------------------------

    best_box = max(
        result.boxes,
        key=lambda box: float(
            box.conf.item()
        ),
    )

    confidence = float(
        best_box.conf.item()
    )

    class_id = int(
        best_box.cls.item()
    )

    class_name = str(
        result.names[class_id]
    )

    x1, y1, x2, y2 = (
        best_box.xyxy[0]
        .detach()
        .cpu()
        .numpy()
        .tolist()
    )


    # --------------------------------------------------------
    # Mettre à jour le tableau principal
    # --------------------------------------------------------

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_detected",
    ] = True

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "number_of_child_detections",
    ] = int(
        len(result.boxes)
    )

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_confidence",
    ] = confidence

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_x1",
    ] = float(x1)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_y1",
    ] = float(y1)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_x2",
    ] = float(x2)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_y2",
    ] = float(y2)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_detection_source",
    ] = "fallback_threshold_0.10"


    # --------------------------------------------------------
    # Sauvegarder l’image annotée du second passage
    # --------------------------------------------------------

    video_fallback_dir = (
        FALLBACK_ANNOTATED_DIR
        / str(
            frame_row["video_id"]
        )
    )

    video_fallback_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    fallback_image_path = (
        video_fallback_dir
        / frame_path.name
    )

    annotated_image = result.plot()

    cv2.imwrite(
        str(fallback_image_path),
        annotated_image,
    )


    fallback_detection_records.append(
        {
            "video_id": str(
                frame_row["video_id"]
            ),

            "true_activity_class": str(
                frame_row[
                    "true_activity_class"
                ]
            ),

            "sample_index": int(
                frame_row["sample_index"]
            ),

            "frame_index": int(
                frame_row["frame_index"]
            ),

            "frame_path": str(
                frame_path
            ),

            "fallback_annotated_path": str(
                fallback_image_path
            ),

            "detected_class_name": class_name,

            "confidence": confidence,

            "bbox_x1": float(x1),
            "bbox_y1": float(y1),
            "bbox_x2": float(x2),
            "bbox_y2": float(y2),

            "detection_source": (
                "fallback_threshold_0.10"
            ),
        }
    )


    del results
    del result
    del annotated_image


# ------------------------------------------------------------
# 7. Créer le résumé final par vidéo
# ------------------------------------------------------------

child_video_summary_fallback_df = (
    child_frame_summary_fallback_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
        ]
    )
    .agg(
        frames_analyzed=(
            "frame_path",
            "size",
        ),

        frames_with_child=(
            "child_detected",
            "sum",
        ),

        mean_child_confidence=(
            "child_confidence",
            "mean",
        ),

        maximum_child_confidence=(
            "child_confidence",
            "max",
        ),

        fallback_detections=(
            "child_detection_source",
            lambda values: int(
                (
                    values
                    == "fallback_threshold_0.10"
                ).sum()
            ),
        ),
    )
    .reset_index()
)


child_video_summary_fallback_df[
    "child_detection_rate"
] = (
    child_video_summary_fallback_df[
        "frames_with_child"
    ]
    /
    child_video_summary_fallback_df[
        "frames_analyzed"
    ]
)


# ------------------------------------------------------------
# 8. Calculer le résultat global
# ------------------------------------------------------------

total_detected_before = int(
    child_frame_summary_df[
        "child_detected"
    ].sum()
)

total_detected_after = int(
    child_frame_summary_fallback_df[
        "child_detected"
    ].sum()
)

new_detections = (
    total_detected_after
    - total_detected_before
)


# ------------------------------------------------------------
# 9. Sauvegarder les nouveaux résultats
# ------------------------------------------------------------

FALLBACK_FRAME_SUMMARY_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_frame_summary_with_fallback.csv"
)

FALLBACK_VIDEO_SUMMARY_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_video_summary_with_fallback.csv"
)

FALLBACK_DETECTIONS_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_fallback_detections.csv"
)


child_frame_summary_fallback_df.to_csv(
    FALLBACK_FRAME_SUMMARY_PATH,
    index=False,
)

child_video_summary_fallback_df.to_csv(
    FALLBACK_VIDEO_SUMMARY_PATH,
    index=False,
)

pd.DataFrame(
    fallback_detection_records
).to_csv(
    FALLBACK_DETECTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Afficher le résumé
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RÉSULTATS APRÈS LE SEUIL DE SECOURS")
print("=" * 80)

display(
    child_video_summary_fallback_df[
        [
            "true_activity_class",
            "frames_analyzed",
            "frames_with_child",
            "child_detection_rate",
            "fallback_detections",
            "mean_child_confidence",
        ]
    ]
)


print(
    "\nDétections avant :",
    f"{total_detected_before} / {EXPECTED_NUMBER_OF_FRAMES}",
)

print(
    "Nouvelles détections obtenues :",
    new_detections,
)

print(
    "Détections après :",
    f"{total_detected_after} / {EXPECTED_NUMBER_OF_FRAMES}",
)

print(
    "Taux global après :",
    f"{100 * total_detected_after / EXPECTED_NUMBER_OF_FRAMES:.2f} %-",
)

print(
    "\nCSV final par frame :",
    FALLBACK_FRAME_SUMMARY_PATH,
)

print(
    "CSV final par vidéo :",
    FALLBACK_VIDEO_SUMMARY_PATH,
)

print("\nCELLULE 5B TERMINÉE")
CHILD_INFERENCE_TOTAL_SECONDS = (
    child_primary_inference_seconds
    + child_fallback_inference_seconds
)
print("Temps total d'inférence YOLO26m :", f"{CHILD_INFERENCE_TOTAL_SECONDS:.3f} s")


# Sauvegarde utile pour la partie 4
child_frame_summary_fallback_df.to_csv(
    MODULE_OUTPUTS_DIR / "child_detection_frames.csv",
    index=False,
)

print("\nCELLULE 3 TERMINÉE")


In [ ]:
# ==============================================================================
# PRÉPARATION ISOLÉE DE VIDEOMAE
# ==============================================================================

# ============================================================
# MODULE 3 — ÉTAPE 2
# PRÉPARATION DU PREMIER VIDEOMAE POUR L'INFÉRENCE
# ============================================================

from pathlib import Path
import shutil
import subprocess
import sys


# ============================================================
# 1. VÉRIFIER QUE LES ÉTAPES PRÉCÉDENTES SONT TERMINÉES
# ============================================================

assert "selected_videos_df" in globals(), (
    "Exécute d'abord les cellules de sélection "
    "des 48 vidéos."
)

assert "extracted_frames_df" in globals(), (
    "Exécute d'abord les cellules d'extraction "
    "des 768 frames."
)

assert len(selected_videos_df) == EXPECTED_NUMBER_OF_VIDEOS
assert len(extracted_frames_df) == EXPECTED_NUMBER_OF_FRAMES


# ============================================================
# 2. RETROUVER LE PREMIER CHECKPOINT VIDEOMAE
# ============================================================

checkpoint_candidates = [
    path
    for path in Path("/kaggle/input").rglob(
        "checkpoint-best.pth"
    )
    if (
        "videomae-final-rhad-result"
        in str(path).lower()
    )
]

print(
    "Checkpoints originaux trouvés :",
    len(checkpoint_candidates),
)

for path in checkpoint_candidates:
    print(" -", path)

assert len(checkpoint_candidates) == 1, (
    "Un seul checkpoint du premier VideoMAE "
    f"était attendu, trouvé : "
    f"{len(checkpoint_candidates)}"
)

VIDEOMAE_CHECKPOINT = (
    checkpoint_candidates[0]
)

assert VIDEOMAE_CHECKPOINT.is_file()

print(
    "\nCheckpoint sélectionné :",
    VIDEOMAE_CHECKPOINT,
)


# ============================================================
# 3. CONFIGURATION DE L'ENVIRONNEMENT ISOLÉ
# ============================================================

VIDEOMAE_ENV_DIR = Path(
    "/kaggle/working/videomae_inference_env"
)

VIDEOMAE_PYTHON = (
    VIDEOMAE_ENV_DIR
    / "bin"
    / "python"
)

VIDEOMAE_REPO_DIR = Path(
    "/kaggle/working/VideoMAEv2"
)

VIDEOMAE_COMMIT = (
    "29eab1e8a588d1b3ec0cdec7b03a86cca491b74b"
)


# ============================================================
# 4. INSTALLER UV
# ============================================================

if shutil.which("uv") is None:

    print(
        "\nInstallation de uv..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "uv",
        ],
        check=True,
    )

UV_EXECUTABLE = shutil.which("uv")

assert UV_EXECUTABLE is not None

print(
    "uv :",
    UV_EXECUTABLE,
)


# ============================================================
# 5. CRÉER L'ENVIRONNEMENT PYTHON 3.10
# ============================================================

if not VIDEOMAE_PYTHON.exists():

    print(
        "\nCréation de l'environnement "
        "VideoMAE Python 3.10..."
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "python",
            "install",
            "3.10",
        ],
        check=True,
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "venv",
            "--python",
            "3.10",
            str(VIDEOMAE_ENV_DIR),
        ],
        check=True,
    )

else:

    print(
        "\nL'environnement VideoMAE existe déjà."
    )

assert VIDEOMAE_PYTHON.exists()

print(
    "Python VideoMAE :",
    VIDEOMAE_PYTHON,
)


# ============================================================
# 6. VÉRIFIER LES DÉPENDANCES
# ============================================================

dependency_test = subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-c",
        (
            "import torch, torchvision, timm, "
            "numpy, decord, cv2, einops, scipy, "
            "pandas, tensorboardX, sklearn"
        ),
    ],
    text=True,
    capture_output=True,
)


# ============================================================
# 7. INSTALLER LES DÉPENDANCES SI NÉCESSAIRE
# ============================================================

if dependency_test.returncode != 0:

    print(
        "\nInstallation de PyTorch..."
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "pip",
            "install",
            "--python",
            str(VIDEOMAE_PYTHON),
            "--index-url",
            "https://download.pytorch.org/whl/cu118",

            "torch==2.1.2",
            "torchvision==0.16.2",
            "torchaudio==2.1.2",
        ],
        check=True,
    )

    print(
        "\nInstallation des dépendances "
        "VideoMAE..."
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "pip",
            "install",
            "--python",
            str(VIDEOMAE_PYTHON),

            "numpy==1.26.4",
            "timm==0.4.12",
            "decord==0.6.0",
            "av",
            "opencv-python-headless",
            "Pillow",
            "einops",
            "scipy",
            "pandas",
            "tensorboardX",
            "matplotlib",
            "scikit-learn",
            "packaging",
        ],
        check=True,
    )

else:

    print(
        "\nLes dépendances VideoMAE "
        "sont déjà installées."
    )


# ============================================================
# 8. TESTER PYTORCH ET LE GPU
# ============================================================

gpu_test_code = r"""
import sys
import torch
import torchvision
import timm
import numpy
import decord

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("timm        :", timm.__version__)
print("NumPy       :", numpy.__version__)
print("Decord      :", decord.__version__)
print("CUDA        :", torch.cuda.is_available())
print("GPU count   :", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA n'est pas disponible."
    )

for index in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {index} :",
        torch.cuda.get_device_name(index)
    )

x = torch.randn(
    64,
    64,
    device="cuda",
)

y = x @ x.T

assert torch.isfinite(y).all()

print("Calcul GPU : OK")
"""

gpu_test_result = subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-c",
        gpu_test_code,
    ],
    text=True,
    capture_output=True,
)

print(
    "\n" + gpu_test_result.stdout
)

if gpu_test_result.stderr:

    print(
        "Messages système :"
    )

    print(
        gpu_test_result.stderr
    )

assert gpu_test_result.returncode == 0, (
    "Le test de l'environnement GPU "
    "VideoMAE a échoué."
)


# ============================================================
# 9. TÉLÉCHARGER LE DÉPÔT VIDEOMAEV2
# ============================================================

if not (
    VIDEOMAE_REPO_DIR
    / ".git"
).exists():

    print(
        "\nTéléchargement de VideoMAEv2..."
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/"
            "OpenGVLab/VideoMAEv2.git",
            str(VIDEOMAE_REPO_DIR),
        ],
        check=True,
    )

else:

    print(
        "\nLe dépôt VideoMAEv2 existe déjà."
    )


# ============================================================
# 10. UTILISER LA VERSION EXACTE DE L'ENTRAÎNEMENT
# ============================================================

subprocess.run(
    [
        "git",
        "checkout",
        VIDEOMAE_COMMIT,
    ],
    cwd=str(VIDEOMAE_REPO_DIR),
    check=True,
)

actual_commit = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=str(VIDEOMAE_REPO_DIR),
    text=True,
).strip()

assert actual_commit == VIDEOMAE_COMMIT

print(
    "Commit VideoMAE :",
    actual_commit,
)


# ============================================================
# 11. VÉRIFIER LES FICHIERS DU DÉPÔT
# ============================================================

required_files = [
    VIDEOMAE_REPO_DIR
    / "run_class_finetuning.py",

    VIDEOMAE_REPO_DIR
    / "engine_for_finetuning.py",

    VIDEOMAE_REPO_DIR
    / "dataset"
    / "build.py",

    VIDEOMAE_REPO_DIR
    / "dataset"
    / "datasets.py",

    VIDEOMAE_REPO_DIR
    / "models"
    / "modeling_finetune.py",
]

for required_file in required_files:

    assert required_file.exists(), (
        f"Fichier absent : "
        f"{required_file}"
    )


# ============================================================
# 12. TESTER LA CRÉATION DE VIDEOMAE
# ============================================================

model_test_code = r"""
import timm
import models

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=12,
    all_frames=16,
    tubelet_size=2,
    drop_path_rate=0.1,
    use_mean_pooling=True,
    init_scale=0.001,
)

classifier = model.get_classifier()

assert classifier.out_features == 12

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Modèle :",
    model.__class__.__name__,
)

print(
    "Paramètres :",
    number_of_parameters,
)

print(
    "Classes :",
    classifier.out_features,
)

print(
    "Création VideoMAE : OK",
)
"""

model_test_result = subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-c",
        model_test_code,
    ],
    cwd=str(VIDEOMAE_REPO_DIR),
    text=True,
    capture_output=True,
)

print(
    "\n" + model_test_result.stdout
)

if model_test_result.stderr:

    print(
        "Messages système du modèle :"
    )

    print(
        model_test_result.stderr
    )

assert model_test_result.returncode == 0, (
    "La création de VideoMAE a échoué."
)


# ============================================================
# 13. RÉSUMÉ FINAL
# ============================================================

print("\n" + "=" * 80)
print("VIDEOMAE PRÊT POUR L'INFÉRENCE")
print("=" * 80)

print(
    "Checkpoint :",
    VIDEOMAE_CHECKPOINT,
)

print(
    "Python :",
    VIDEOMAE_PYTHON,
)

print(
    "Dépôt :",
    VIDEOMAE_REPO_DIR,
)

print(
    "Commit :",
    actual_commit,
)

print(
    "\n PRÉPARATION VIDEOMAE TERMINÉE"
)
print("\nCELLULE 4 TERMINÉE")


In [ ]:
# ============================================================================
# CELLULE 5 — INFÉRENCE VIDEOMAE SUR LES VIDÉOS E2E
# ============================================================================

from pathlib import Path
import os
import shutil
import subprocess
import time

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------
# 1. Configuration et mapping exact des 12 classes
# ----------------------------------------------------------------------------

VIDEOMAE_CLASS_NAMES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

VIDEOMAE_CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(VIDEOMAE_CLASS_NAMES)
}


assert (
    VIDEOMAE_CLASS_TO_INDEX["normal"]
    ==
    NORMAL_CLASS_INDEX
    ==
    7
)

assert Path(VIDEOMAE_PYTHON).exists(), VIDEOMAE_PYTHON
assert VIDEOMAE_REPO_DIR.exists(), VIDEOMAE_REPO_DIR
assert VIDEOMAE_CHECKPOINT.exists(), VIDEOMAE_CHECKPOINT


print("=" * 80)
print("VIDEOMAE — PRÉPARATION E2E")
print("=" * 80)

print(
    "Checkpoint :",
    VIDEOMAE_CHECKPOINT
)

print(
    "Nombre de classes du modèle :",
    len(VIDEOMAE_CLASS_NAMES)
)


# ----------------------------------------------------------------------------
# 2. Adapter le dépôt à RHAD, sans DeepSpeed obligatoire
# ----------------------------------------------------------------------------

RUN_FILE = (
    VIDEOMAE_REPO_DIR
    /
    "run_class_finetuning.py"
)

BUILD_FILE = (
    VIDEOMAE_REPO_DIR
    /
    "dataset"
    /
    "build.py"
)


# ----------------------------------------------------------------------------
# 2.1 Patch DeepSpeed
# ----------------------------------------------------------------------------

run_text = RUN_FILE.read_text(
    encoding="utf-8"
)


if (
    "try:\n    import deepspeed"
    not in run_text
):

    assert (
        "import deepspeed"
        in run_text
    ), (
        "Import deepspeed introuvable."
    )

    run_text = run_text.replace(

        "import deepspeed",

        """try:
    import deepspeed
except ImportError:
    deepspeed = None""",

        1,
    )


old_deepspeed_block = """    if known_args.enable_deepspeed:
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""


new_deepspeed_block = """    if known_args.enable_deepspeed:
        if deepspeed is None:
            raise ImportError("DeepSpeed demandé mais non installé.")
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""


if old_deepspeed_block in run_text:

    run_text = run_text.replace(

        old_deepspeed_block,

        new_deepspeed_block,

        1,
    )


# ----------------------------------------------------------------------------
# 2.2 Ajouter RHAD dans --data_set
# ----------------------------------------------------------------------------

argument_start = run_text.find(
    "parser.add_argument(\n        '--data_set'"
)

assert argument_start != -1, (
    "Argument --data_set introuvable."
)


argument_end = run_text.find(
    "type=str",
    argument_start,
)

assert argument_end != -1


argument_block = run_text[
    argument_start:
    argument_end
]


if "'RHAD'" not in argument_block:

    assert "'MIT'" in argument_block

    patched_argument_block = (
        argument_block.replace(
            "'MIT'",
            "'MIT', 'RHAD'",
            1,
        )
    )

    run_text = (
        run_text[:argument_start]
        +
        patched_argument_block
        +
        run_text[argument_end:]
    )


RUN_FILE.write_text(
    run_text,
    encoding="utf-8",
)


# ----------------------------------------------------------------------------
# 2.3 Utiliser test.csv lors du test
# ----------------------------------------------------------------------------

build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)


old_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'val.csv')
"""


new_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'test.csv')
"""


if old_test_block in build_text:

    build_text = build_text.replace(

        old_test_block,

        new_test_block,

        1,
    )


# ----------------------------------------------------------------------------
# 2.4 Ajouter la branche RHAD
# ----------------------------------------------------------------------------

rhad_branch = """
    elif args.data_set == 'RHAD':
        dataset = VideoClsDataset(
            anno_path=anno_path,
            data_root=args.data_root,
            mode=mode,
            clip_len=args.num_frames,
            frame_sample_rate=args.sampling_rate,
            num_segment=1,
            test_num_segment=args.test_num_segment,
            test_num_crop=args.test_num_crop,
            num_crop=1 if not test_mode else 3,
            keep_aspect_ratio=True,
            crop_size=args.input_size,
            short_side_size=args.short_side_size,
            new_height=256,
            new_width=320,
            sparse_sample=False,
            args=args,
        )
        nb_classes = 12
"""


unsupported_block = """    else:
        raise NotImplementedError('Unsupported Dataset')
"""


if (
    "elif args.data_set == 'RHAD':"
    not in build_text
):

    assert (
        unsupported_block
        in build_text
    )

    build_text = build_text.replace(

        unsupported_block,

        rhad_branch
        +
        "\n"
        +
        unsupported_block,

        1,
    )


BUILD_FILE.write_text(
    build_text,
    encoding="utf-8",
)


# ----------------------------------------------------------------------------
# 2.5 Vérification syntaxique
# ----------------------------------------------------------------------------

subprocess.run(

    [
        str(VIDEOMAE_PYTHON),
        "-m",
        "py_compile",
        str(RUN_FILE),
        str(BUILD_FILE),
    ],

    check=True,
)


print(
    "Dépôt VideoMAE préparé"
)


# ----------------------------------------------------------------------------
# 3. Préparer UNIQUEMENT les vidéos du test E2E
# ----------------------------------------------------------------------------

videomae_selected_df = (

    extracted_frames_df[
        [
            "video_id",
            "class_name",
            "video_path",
        ]
    ]

    .drop_duplicates(
        subset=[
            "video_id"
        ]
    )

    .sort_values(
        "video_id"
    )

    .reset_index(
        drop=True
    )
)


# ----------------------------------------------------------------------------
# Vérifications E2E
# ----------------------------------------------------------------------------

assert (
    len(
        videomae_selected_df
    )
    ==
    EXPECTED_NUMBER_OF_VIDEOS
), (
    f"{EXPECTED_NUMBER_OF_VIDEOS} vidéos attendues, "
    f"{len(videomae_selected_df)} trouvées."
)


assert (
    videomae_selected_df[
        "video_id"
    ].nunique()
    ==
    EXPECTED_NUMBER_OF_VIDEOS
)


# IMPORTANT :
# Le test E2E utilise seulement 5 classes représentatives.
# On vérifie donc que ces classes appartiennent aux 12 classes
# connues par VideoMAE, et NON que les 12 classes sont présentes.

classes_e2e = set(
    videomae_selected_df[
        "class_name"
    ].astype(str)
)


assert (
    classes_e2e
    .issubset(
        set(
            VIDEOMAE_CLASS_NAMES
        )
    )
), (
    "Une ou plusieurs classes E2E "
    "ne sont pas reconnues par VideoMAE : "
    f"{sorted(classes_e2e)}"
)


print(
    "\nClasses présentes dans le test E2E :"
)

for class_name in sorted(
    classes_e2e
):

    print(
        " -",
        class_name
    )


print(
    "\nNombre de vidéos E2E :",
    len(
        videomae_selected_df
    )
)


print(
    "Sélection E2E VideoMAE valide"
)


# ----------------------------------------------------------------------------
# 4. Créer les dossiers d'inférence
# ----------------------------------------------------------------------------

VIDEOMAE_INFERENCE_DIR = Path(
    "/kaggle/working/E2E_videomae_inference"
)

VIDEOMAE_VIDEO_ROOT = (
    VIDEOMAE_INFERENCE_DIR
    /
    "videos"
)

VIDEOMAE_ANNOTATIONS_DIR = (
    VIDEOMAE_INFERENCE_DIR
    /
    "annotations"
)

VIDEOMAE_RAW_OUTPUT_DIR = (
    VIDEOMAE_INFERENCE_DIR
    /
    "raw_predictions"
)


if VIDEOMAE_INFERENCE_DIR.exists():

    shutil.rmtree(
        VIDEOMAE_INFERENCE_DIR
    )


for directory in [

    VIDEOMAE_VIDEO_ROOT,

    VIDEOMAE_ANNOTATIONS_DIR,

    VIDEOMAE_RAW_OUTPUT_DIR,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ----------------------------------------------------------------------------
# 5. Créer les annotations VideoMAE
# ----------------------------------------------------------------------------

annotation_lines = []

inference_records = []


for _, row in (
    videomae_selected_df
    .iterrows()
):

    video_id = str(
        row[
            "video_id"
        ]
    )


    true_class = str(
        row[
            "class_name"
        ]
    )


    source_path = Path(
        row[
            "video_path"
        ]
    )


    assert (
        source_path.exists()
    ), source_path


    assert (
        true_class
        in
        VIDEOMAE_CLASS_TO_INDEX
    )


    label = (
        VIDEOMAE_CLASS_TO_INDEX[
            true_class
        ]
    )


    alias_filename = (
        f"{video_id}"
        f"{source_path.suffix.lower()}"
    )


    alias_path = (
        VIDEOMAE_VIDEO_ROOT
        /
        alias_filename
    )


    # --------------------------------------------------------
    # Symlink vers la vidéo originale
    # --------------------------------------------------------

    os.symlink(
        str(
            source_path
        ),
        str(
            alias_path
        ),
    )


    annotation_lines.append(

        f"{alias_filename} "
        f"{label}\n"
    )


    inference_records.append(

        {
            "video_id":
                video_id,

            "alias_filename":
                alias_filename,

            "video_path":
                str(
                    source_path
                ),

            "true_label":
                int(
                    label
                ),

            "true_class":
                true_class,
        }
    )


videomae_inference_index_df = (
    pd.DataFrame(
        inference_records
    )
)


assert (
    len(
        videomae_inference_index_df
    )
    ==
    EXPECTED_NUMBER_OF_VIDEOS
)


for filename in [

    "train.csv",

    "val.csv",

    "test.csv",

]:

    (
        VIDEOMAE_ANNOTATIONS_DIR
        /
        filename
    ).write_text(

        "".join(
            annotation_lines
        ),

        encoding="utf-8",
    )


videomae_inference_index_df.to_csv(

    VIDEOMAE_INFERENCE_DIR
    /
    "videomae_inference_index.csv",

    index=False,
)


print(
    "\nAnnotations VideoMAE créées :",
    len(
        annotation_lines
    )
)


# ----------------------------------------------------------------------------
# 6. Exécuter le checkpoint original
# ----------------------------------------------------------------------------

command = [

    str(
        VIDEOMAE_PYTHON
    ),

    "-m",

    "torch.distributed.run",

    "--nproc_per_node=2",

    "--master_port=29621",

    "run_class_finetuning.py",

    "--model",
    "vit_base_patch16_224",

    "--nb_classes",
    "12",

    "--use_mean_pooling",

    "--init_scale",
    "0.001",

    "--data_set",
    "RHAD",

    "--data_path",
    str(
        VIDEOMAE_ANNOTATIONS_DIR
    ),

    "--data_root",
    str(
        VIDEOMAE_VIDEO_ROOT
    ),

    "--input_size",
    "224",

    "--short_side_size",
    "224",

    "--num_frames",
    "16",

    "--num_segments",
    "1",

    "--tubelet_size",
    "2",

    "--sampling_rate",
    "4",

    "--batch_size",
    "2",

    "--num_workers",
    "2",

    "--seed",
    "42",

    "--test_num_segment",
    "2",

    "--test_num_crop",
    "3",

    "--dist_eval",

    "--finetune",
    str(
        VIDEOMAE_CHECKPOINT
    ),

    "--model_key",
    "model",

    "--no_auto_resume",

    "--eval",

    "--output_dir",
    str(
        VIDEOMAE_RAW_OUTPUT_DIR
    ),

    "--log_dir",
    str(
        VIDEOMAE_RAW_OUTPUT_DIR
    ),
]


environment = (
    os.environ.copy()
)

environment[
    "CUDA_VISIBLE_DEVICES"
] = "0,1"

environment[
    "OMP_NUM_THREADS"
] = "1"

environment[
    "PYTHONUNBUFFERED"
] = "1"


log_path = (
    VIDEOMAE_INFERENCE_DIR
    /
    "videomae_inference_console.log"
)


print(
    "\n"
    + "=" * 80
)

print(
    "INFÉRENCE VIDEOMAE — TEST E2E"
)

print(
    "=" * 80
)


print(
    "Checkpoint :",
    VIDEOMAE_CHECKPOINT
)


print(
    "Vidéos :",
    len(
        videomae_inference_index_df
    )
)


print(
    "Vues par vidéo : 6 "
    "(2 segments × 3 crops)"
)


start_time = (
    time.perf_counter()
)


with log_path.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(

        command,

        cwd=str(
            VIDEOMAE_REPO_DIR
        ),

        env=environment,

        stdout=subprocess.PIPE,

        stderr=subprocess.STDOUT,

        text=True,

        bufsize=1,
    )


    assert (
        process.stdout
        is not None
    )


    for line in (
        process.stdout
    ):

        print(
            line,
            end="",
            flush=True,
        )

        log_file.write(
            line
        )

        log_file.flush()


    return_code = (
        process.wait()
    )


total_inference_time = (

    time.perf_counter()
    -
    start_time
)


if (
    return_code
    != 0
):

    raise RuntimeError(

        "L'inférence VideoMAE a échoué. "
        f"Code de retour : {return_code}. "
        f"Consulter : {log_path}"
    )


print(
    "\nInférence VideoMAE terminée"
)


# ----------------------------------------------------------------------------
# 7. Lire et agréger les 6 vues de chaque vidéo
# ----------------------------------------------------------------------------

raw_records = []


raw_files = [

    VIDEOMAE_RAW_OUTPUT_DIR
    /
    "0.txt",

    VIDEOMAE_RAW_OUTPUT_DIR
    /
    "1.txt",

]


assert all(
    path.exists()
    for path in raw_files
), (
    "Prédictions brutes VideoMAE absentes."
)


for raw_file in raw_files:

    lines = (
        raw_file.read_text(

            encoding="utf-8",

            errors="ignore",

        )
        .splitlines()
    )


    for line in lines:

        if (
            "["
            not in line
            or
            "]"
            not in line
        ):

            continue


        video_part, remaining = (
            line.split(
                "[",
                1,
            )
        )


        logits_text, metadata_text = (
            remaining.split(
                "]",
                1,
            )
        )


        logits = (
            np.fromstring(

                logits_text,

                dtype=np.float64,

                sep=",",
            )
        )


        metadata = (
            metadata_text
            .strip()
            .split()
        )


        if (
            len(logits)
            != 12
            or
            len(metadata)
            < 3
        ):

            continue


        record = {

            "video_id":
                Path(
                    video_part.strip()
                ).stem,

            "true_label":
                int(
                    metadata[0]
                ),

            "segment_index":
                int(
                    metadata[1]
                ),

            "crop_index":
                int(
                    metadata[2]
                ),
        }


        for class_index in range(
            12
        ):

            record[
                f"logit_{class_index}"
            ] = float(
                logits[
                    class_index
                ]
            )


        raw_records.append(
            record
        )


videomae_raw_predictions_df = (
    pd.DataFrame(
        raw_records
    )
)


expected_views = (

    EXPECTED_NUMBER_OF_VIDEOS

    *

    2

    *

    3
)


print(
    "\nVues VideoMAE récupérées :",
    len(
        videomae_raw_predictions_df
    ),
    "/",
    expected_views
)


assert (
    len(
        videomae_raw_predictions_df
    )
    ==
    expected_views
), (

    f"{expected_views} vues attendues, "
    f"{len(videomae_raw_predictions_df)} trouvées."
)


# ----------------------------------------------------------------------------
# 8. Softmax stable
# ----------------------------------------------------------------------------

def stable_softmax(
    logits
):

    logits = np.asarray(
        logits,
        dtype=np.float64,
    )


    logits = (
        logits
        -
        np.max(
            logits
        )
    )


    values = np.exp(
        logits
    )


    return (
        values
        /
        values.sum()
    )


# ----------------------------------------------------------------------------
# 9. Agrégation des prédictions
# ----------------------------------------------------------------------------

logit_columns = [

    f"logit_{index}"

    for index in range(
        12
    )
]


prediction_records = []


for _, reference_row in (
    videomae_inference_index_df
    .iterrows()
):

    video_id = str(
        reference_row[
            "video_id"
        ]
    )


    video_views = (

        videomae_raw_predictions_df[

            videomae_raw_predictions_df[
                "video_id"
            ]
            ==
            video_id
        ]
    )


    assert (
        len(
            video_views
        )
        ==
        6
    ), (

        f"{video_id} : "
        f"{len(video_views)} vues "
        "au lieu de 6."
    )


    mean_logits = (

        video_views[
            logit_columns
        ]

        .to_numpy(
            dtype=np.float64
        )

        .mean(
            axis=0
        )
    )


    probabilities = (
        stable_softmax(
            mean_logits
        )
    )


    predicted_label = int(
        np.argmax(
            probabilities
        )
    )


    predicted_class = (

        VIDEOMAE_CLASS_NAMES[
            predicted_label
        ]
    )


    confidence = float(

        probabilities[
            predicted_label
        ]
    )


    is_abnormal = (

        predicted_class
        !=
        "normal"
    )


    true_class = str(
        reference_row[
            "true_class"
        ]
    )


    prediction_records.append(

        {
            "video_id":
                video_id,

            "video_path":
                str(
                    reference_row[
                        "video_path"
                    ]
                ),

            "reference_class":
                true_class,

            "predicted_label":
                predicted_label,

            "predicted_class":
                predicted_class,

            "confidence":
                confidence,

            "is_abnormal":
                bool(
                    is_abnormal
                ),

            "correct_prediction":
                bool(
                    predicted_class
                    ==
                    true_class
                ),
        }
    )


# ----------------------------------------------------------------------------
# 10. DataFrame final VideoMAE
# ----------------------------------------------------------------------------

videomae_predictions_df = (
    pd.DataFrame(
        prediction_records
    )
)


assert (
    len(
        videomae_predictions_df
    )
    ==
    EXPECTED_NUMBER_OF_VIDEOS
)


assert (
    videomae_predictions_df[
        "video_id"
    ].nunique()
    ==
    EXPECTED_NUMBER_OF_VIDEOS
)


assert (
    videomae_predictions_df[
        "confidence"
    ]
    .between(
        0,
        1,
    )
    .all()
)


mean_time_per_video = (

    total_inference_time

    /

    len(
        videomae_predictions_df
    )
)


videomae_predictions_df[
    "mean_processing_time_per_video_seconds"
] = float(
    mean_time_per_video
)


# ----------------------------------------------------------------------------
# 11. Ajouter le split
# ----------------------------------------------------------------------------

videomae_predictions_df = (

    videomae_predictions_df.merge(

        selected_videos_df[
            [
                "video_id",
                "split",
            ]
        ],

        on="video_id",

        how="left",

        validate="one_to_one",
    )
)


assert (
    videomae_predictions_df[
        "split"
    ]
    .notna()
    .all()
)


# ----------------------------------------------------------------------------
# 12. Sauvegarde
# ----------------------------------------------------------------------------

VIDEOMAE_PREDICTIONS_PATH = (

    MODULE_OUTPUTS_DIR

    /

    "videomae_predictions.csv"
)


videomae_predictions_df.to_csv(

    VIDEOMAE_PREDICTIONS_PATH,

    index=False,
)


# ----------------------------------------------------------------------------
# 13. Résultats
# ----------------------------------------------------------------------------

print(
    "\n"
    + "=" * 80
)

print(
    "VIDEOMAE — RÉSULTATS E2E"
)

print(
    "=" * 80
)


display(

    videomae_predictions_df[
        [
            "video_id",
            "reference_class",
            "predicted_class",
            "confidence",
            "is_abnormal",
            "correct_prediction",
        ]
    ]
)


print(
    "\nPrédictions correctes :",
    int(
        videomae_predictions_df[
            "correct_prediction"
        ].sum()
    ),
    "/",
    EXPECTED_NUMBER_OF_VIDEOS,
)


print(
    "Temps total :",
    round(
        total_inference_time,
        3,
    ),
    "s",
)


print(
    "Temps moyen par vidéo :",
    round(
        mean_time_per_video,
        3,
    ),
    "s",
)


print(
    "CSV :",
    VIDEOMAE_PREDICTIONS_PATH
)


print(
    "\nCELLULE 5 TERMINÉE"
)

In [ ]:
# ==============================================================================
# DÉTECTION CONTEXTUELLE OPEN-VOCABULARY AVEC YOLOE-26s
# ==============================================================================

import gc
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLOE


# ------------------------------------------------------------------------------
# 1. Configuration issue du benchmark
# ------------------------------------------------------------------------------

YOLOE_MODEL_NAME = "YOLOE-26s"
YOLOE_MODEL_ID = "yoloe-26s-seg.pt"
YOLOE_IMAGE_SIZE = 640
YOLOE_NMS_IOU = 0.50
YOLOE_MAX_DETECTIONS = 500

# Ces 43 classes sont exactement celles représentées dans les annotations
# du benchmark. La classe child est absente car l'enfant est déjà traité
# par le détecteur spécialisé YOLO26m.
YOLOE_CONTEXT_CLASSES = [
    "bed",
    "bench",
    "bookshelf",
    "bottle",
    "box",
    "cabinet",
    "cable",
    "chair",
    "cup",
    "cupboard",
    "curtain",
    "curtain_rod",
    "door",
    "dresser",
    "electrical_plug",
    "fan",
    "food_piece",
    "keys",
    "kitchen_utensil",
    "knife",
    "knife_block",
    "lamp",
    "marker",
    "medicine",
    "metal_bars",
    "oven",
    "pen",
    "railing",
    "refrigerator",
    "remote_control",
    "rug",
    "scissors",
    "shelf",
    "shoe",
    "sofa",
    "stairs",
    "table",
    "teddy_bear",
    "television",
    "television_stand",
    "toy",
    "toy_car",
    "wall",
]

YOLOE_TEXT_PROMPTS = [
    class_name.replace("_", " ")
    for class_name in YOLOE_CONTEXT_CLASSES
]

PROMPT_TO_CLASS = dict(
    zip(YOLOE_TEXT_PROMPTS, YOLOE_CONTEXT_CLASSES)
)

YOLOE_OBJECTS_DIR = PROJECT_DIR / "yoloe_26s_objects"
YOLOE_OBJECTS_DIR.mkdir(parents=True, exist_ok=True)

YOLOE_INFERENCE_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_inference_detections.csv"
)

YOLOE_LATENCY_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_frame_latency.csv"
)

assert OBJECT_CONFIDENCE_THRESHOLD == 0.14
assert len(YOLOE_CONTEXT_CLASSES) == 43
assert len(YOLOE_CONTEXT_CLASSES) == len(set(YOLOE_CONTEXT_CLASSES))
assert len(extracted_frames_df) == EXPECTED_NUMBER_OF_FRAMES


# ------------------------------------------------------------------------------
# 2. NMS identique au protocole du benchmark
# ------------------------------------------------------------------------------

def box_iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    intersection_x1 = max(ax1, bx1)
    intersection_y1 = max(ay1, by1)
    intersection_x2 = min(ax2, bx2)
    intersection_y2 = min(ay2, by2)

    intersection = (
        max(0.0, intersection_x2 - intersection_x1)
        * max(0.0, intersection_y2 - intersection_y1)
    )

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def class_aware_nms(records, iou_threshold=YOLOE_NMS_IOU):
    kept_records = []
    records_by_class = defaultdict(list)

    for record in records:
        records_by_class[
            record["detected_class_name"]
        ].append(record)

    for class_records in records_by_class.values():
        remaining = sorted(
            class_records,
            key=lambda item: item["confidence"],
            reverse=True,
        )

        while remaining:
            best_record = remaining.pop(0)
            kept_records.append(best_record)

            best_box = [
                best_record[key]
                for key in (
                    "bbox_x1",
                    "bbox_y1",
                    "bbox_x2",
                    "bbox_y2",
                )
            ]

            remaining = [
                candidate
                for candidate in remaining
                if box_iou_xyxy(
                    best_box,
                    [
                        candidate[key]
                        for key in (
                            "bbox_x1",
                            "bbox_y1",
                            "bbox_x2",
                            "bbox_y2",
                        )
                    ],
                )
                < iou_threshold
            ]

    return kept_records


# ------------------------------------------------------------------------------
# 3. Chargement et configuration du modèle
# ------------------------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()

object_model = YOLOE(YOLOE_MODEL_ID)
object_model.set_classes(YOLOE_TEXT_PROMPTS)

YOLOE_DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("DÉTECTEUR CONTEXTUEL")
print("=" * 80)
print("Modèle :", YOLOE_MODEL_NAME)
print("Checkpoint :", YOLOE_MODEL_ID)
print("Classes recherchées :", len(YOLOE_CONTEXT_CLASSES))
print(
    "Seuil de confiance calibré :",
    OBJECT_CONFIDENCE_THRESHOLD,
)
print("NMS IoU :", YOLOE_NMS_IOU)
print("Périphérique :", YOLOE_DEVICE)


# ------------------------------------------------------------------------------
# 4. Préchauffage GPU non inclus dans les temps mesurés
# ------------------------------------------------------------------------------

ordered_object_frames_df = (
    extracted_frames_df
    .sort_values(
        ["class_name", "video_id", "sample_index"]
    )
    .reset_index(drop=True)
)

warmup_frame_path = Path(
    ordered_object_frames_df.iloc[0]["frame_path"]
)

assert warmup_frame_path.exists()

object_model.predict(
    source=str(warmup_frame_path),
    conf=OBJECT_CONFIDENCE_THRESHOLD,
    iou=0.99,
    agnostic_nms=False,
    imgsz=YOLOE_IMAGE_SIZE,
    max_det=YOLOE_MAX_DETECTIONS,
    device=YOLOE_DEVICE,
    half=torch.cuda.is_available(),
    verbose=False,
)


# ------------------------------------------------------------------------------
# 5. Inférence sur les frames E2E
# ------------------------------------------------------------------------------

yoloe_detection_records = []
yoloe_latency_records = []

for frame_row in tqdm(
    ordered_object_frames_df.itertuples(index=False),
    total=len(ordered_object_frames_df),
    desc="Détection YOLOE-26s",
):
    frame_path = Path(frame_row.frame_path)

    assert frame_path.exists(), (
        f"Frame absente : {frame_path}"
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    result = object_model.predict(
        source=str(frame_path),
        conf=OBJECT_CONFIDENCE_THRESHOLD,
        iou=0.99,
        agnostic_nms=False,
        imgsz=YOLOE_IMAGE_SIZE,
        max_det=YOLOE_MAX_DETECTIONS,
        device=YOLOE_DEVICE,
        half=torch.cuda.is_available(),
        verbose=False,
    )[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed_seconds = time.perf_counter() - start_time

    yoloe_latency_records.append(
        {
            "video_id": str(frame_row.video_id),
            "sample_index": int(frame_row.sample_index),
            "frame_index": int(frame_row.frame_index),
            "frame_path": str(frame_path),
            "latency_seconds": float(elapsed_seconds),
        }
    )

    frame_records = []

    if result.boxes is not None and len(result.boxes) > 0:
        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confidences = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        class_indices = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(int)
        )

        for box, confidence, class_index in zip(
            boxes,
            confidences,
            class_indices,
        ):
            if not 0 <= class_index < len(
                YOLOE_CONTEXT_CLASSES
            ):
                continue

            class_name = YOLOE_CONTEXT_CLASSES[
                class_index
            ]

            x1, y1, x2, y2 = map(float, box)

            # Limitation aux dimensions de la frame.
            x1 = max(
                0.0,
                min(float(frame_row.width), x1),
            )
            y1 = max(
                0.0,
                min(float(frame_row.height), y1),
            )
            x2 = max(
                0.0,
                min(float(frame_row.width), x2),
            )
            y2 = max(
                0.0,
                min(float(frame_row.height), y2),
            )

            if x2 <= x1 or y2 <= y1:
                continue

            frame_records.append(
                {
                    "video_id": str(frame_row.video_id),
                    "true_activity_class": str(
                        frame_row.class_name
                    ),
                    "sample_index": int(
                        frame_row.sample_index
                    ),
                    "frame_index": int(
                        frame_row.frame_index
                    ),
                    "timestamp_seconds": float(
                        frame_row.timestamp_seconds
                    ),
                    "frame_path": str(frame_path),
                    "detected_class_name": class_name,
                    "confidence": float(confidence),
                    "bbox_x1": x1,
                    "bbox_y1": y1,
                    "bbox_x2": x2,
                    "bbox_y2": y2,
                    "detector_model": YOLOE_MODEL_NAME,
                }
            )

    # Suppression des détections redondantes d'une même classe.
    frame_records = class_aware_nms(
        frame_records,
        iou_threshold=YOLOE_NMS_IOU,
    )

    for detection_index, record in enumerate(
        frame_records
    ):
        record["detection_index"] = int(
            detection_index
        )
        yoloe_detection_records.append(record)


# ------------------------------------------------------------------------------
# 6. Création et sauvegarde des résultats
# ------------------------------------------------------------------------------

YOLOE_COLUMNS = [
    "video_id",
    "true_activity_class",
    "sample_index",
    "frame_index",
    "timestamp_seconds",
    "frame_path",
    "detection_index",
    "detected_class_name",
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
    "detector_model",
]

yoloe_inference_df = pd.DataFrame(
    yoloe_detection_records,
    columns=YOLOE_COLUMNS,
)

yoloe_latency_df = pd.DataFrame(
    yoloe_latency_records
)

yoloe_inference_df.to_csv(
    YOLOE_INFERENCE_PATH,
    index=False,
)

yoloe_latency_df.to_csv(
    YOLOE_LATENCY_PATH,
    index=False,
)


# ------------------------------------------------------------------------------
# 7. Vérifications minimales
# ------------------------------------------------------------------------------

assert len(yoloe_latency_df) == EXPECTED_NUMBER_OF_FRAMES
assert not yoloe_inference_df.empty

assert yoloe_inference_df["confidence"].ge(
    OBJECT_CONFIDENCE_THRESHOLD
).all()

assert set(
    yoloe_inference_df["detected_class_name"]
).issubset(
    set(YOLOE_CONTEXT_CLASSES)
)

assert (
    yoloe_inference_df["bbox_x2"]
    > yoloe_inference_df["bbox_x1"]
).all()

assert (
    yoloe_inference_df["bbox_y2"]
    > yoloe_inference_df["bbox_y1"]
).all()


# ------------------------------------------------------------------------------
# 8. Résumé
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("INFÉRENCE YOLOE-26s TERMINÉE")
print("=" * 80)

print(
    "Frames traitées :",
    len(yoloe_latency_df),
)

print(
    "Détections après NMS :",
    len(yoloe_inference_df),
)

print(
    "Frames avec au moins un objet :",
    yoloe_inference_df["frame_path"].nunique(),
)

print(
    "Classes différentes détectées :",
    yoloe_inference_df[
        "detected_class_name"
    ].nunique(),
)

print(
    "Latence moyenne par frame :",
    f"{1000 * yoloe_latency_df['latency_seconds'].mean():.2f} ms",
)

print(
    "CSV des détections :",
    YOLOE_INFERENCE_PATH,
)

print(
    "CSV des latences :",
    YOLOE_LATENCY_PATH,
)


# ------------------------------------------------------------------------------
# 9. Libération de la mémoire GPU
# ------------------------------------------------------------------------------

del object_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

yoloe_inference_df.to_csv(
    MODULE_OUTPUTS_DIR / "yoloe_context_detections.csv",
    index=False,
)

print("\nCELLULE 6 TERMINÉE")


In [ ]:
# ==============================================================================
# CONTRÔLE QUALITÉ ET PRÉPARATION DES DÉTECTIONS YOLOE-26s
# ==============================================================================

from IPython.display import display


if "yoloe_inference_df" not in globals():
    yoloe_inference_df = pd.read_csv(YOLOE_INFERENCE_PATH)

required_object_columns = [
    "video_id",
    "true_activity_class",
    "sample_index",
    "frame_index",
    "frame_path",
    "detected_class_name",
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
]

missing_object_columns = [
    column
    for column in required_object_columns
    if column not in yoloe_inference_df.columns
]
assert not missing_object_columns, (
    f"Colonnes YOLOE manquantes : {missing_object_columns}"
)

object_detections_df = yoloe_inference_df.copy()

numeric_columns = [
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
]
for column in numeric_columns:
    object_detections_df[column] = pd.to_numeric(
        object_detections_df[column],
        errors="coerce",
    )

# Le nettoyage retire uniquement les données techniquement invalides. Aucun
# filtrage temporel arbitraire n'est appliqué : un objet dangereux visible sur
# une seule frame doit rester disponible pour le module d'évaluation du risque.
object_detections_df = (
    object_detections_df
    .dropna(subset=required_object_columns)
    .loc[
        lambda frame: (
            frame["confidence"].ge(OBJECT_CONFIDENCE_THRESHOLD)
            & frame["confidence"].le(1.0)
            & frame["bbox_x2"].gt(frame["bbox_x1"])
            & frame["bbox_y2"].gt(frame["bbox_y1"])
            & frame["detected_class_name"].isin(YOLOE_CONTEXT_CLASSES)
        )
    ]
    .drop_duplicates(
        subset=[
            "video_id",
            "sample_index",
            "detected_class_name",
            "bbox_x1",
            "bbox_y1",
            "bbox_x2",
            "bbox_y2",
        ]
    )
    .sort_values(
        [
            "video_id",
            "sample_index",
            "confidence",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

object_detections_df["bbox_width"] = (
    object_detections_df["bbox_x2"]
    - object_detections_df["bbox_x1"]
)
object_detections_df["bbox_height"] = (
    object_detections_df["bbox_y2"]
    - object_detections_df["bbox_y1"]
)
object_detections_df["detector_model"] = YOLOE_MODEL_NAME

# Résumé temporel : les absences ponctuelles dans certaines frames ne font pas
# disparaître l'objet du résumé vidéo. Le nombre de frames de présence est
# conservé pour le JSON final.
yoloe_temporal_summary_df = (
    object_detections_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
            "detected_class_name",
        ],
        as_index=False,
    )
    .agg(
        frames_present=("sample_index", "nunique"),
        detections_count=("detected_class_name", "size"),
        confidence_mean=("confidence", "mean"),
        confidence_max=("confidence", "max"),
    )
)
yoloe_temporal_summary_df["presence_ratio"] = (
    yoloe_temporal_summary_df["frames_present"]
    / NUM_CONTEXT_FRAMES
)

YOLOE_FINAL_DETECTIONS_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_final_detections.csv"
)
YOLOE_TEMPORAL_SUMMARY_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_temporal_summary.csv"
)

object_detections_df.to_csv(
    YOLOE_FINAL_DETECTIONS_PATH,
    index=False,
)
yoloe_temporal_summary_df.to_csv(
    YOLOE_TEMPORAL_SUMMARY_PATH,
    index=False,
)

# Alias utilisé par la cellule d'export final.
FINAL_DETECTIONS_PATH = YOLOE_FINAL_DETECTIONS_PATH

assert not object_detections_df.empty
assert object_detections_df["confidence"].between(
    OBJECT_CONFIDENCE_THRESHOLD,
    1.0,
).all()
assert (object_detections_df["bbox_width"] > 0).all()
assert (object_detections_df["bbox_height"] > 0).all()
assert set(object_detections_df["detected_class_name"]).issubset(
    set(YOLOE_CONTEXT_CLASSES)
)

print("=" * 80)
print("CONTRÔLE QUALITÉ YOLOE-26s TERMINÉ")
print("=" * 80)
print("Détections retenues :", len(object_detections_df))
print(
    "Frames contenant au moins un objet :",
    object_detections_df["frame_path"].nunique(),
    f"/ {EXPECTED_NUMBER_OF_FRAMES}",
)
print(
    "Classes détectées :",
    object_detections_df["detected_class_name"].nunique(),
)
print("Détections finales :", YOLOE_FINAL_DETECTIONS_PATH)
print("Résumé temporel :", YOLOE_TEMPORAL_SUMMARY_PATH)

display(
    object_detections_df[
        "detected_class_name"
    ].value_counts().head(20)
)

print("\nCELLULE 7 TERMINÉE")


In [ ]:
# ==============================================================================
# CELLULE 7 — FUSION DESCRIPTIVE ENFANT–OBJETS
# Géométrie continue, sans décision near/far
# ==============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ==============================================================================
# 1. DOSSIERS ET FICHIERS DE SORTIE
# ==============================================================================

GEOMETRY_OUTPUT_DIR = (
    PROJECT_DIR
    / "child_object_geometry"
)

GEOMETRY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHILD_OBJECT_GEOMETRY_CSV_PATH = (
    GEOMETRY_OUTPUT_DIR
    / "child_object_geometry.csv"
)

OBJECT_TEMPORAL_SUMMARY_CSV_PATH = (
    GEOMETRY_OUTPUT_DIR
    / "object_temporal_summary.csv"
)

VIDEO_CONTEXT_SUMMARY_CSV_PATH = (
    GEOMETRY_OUTPUT_DIR
    / "video_context_summary.csv"
)


# ==============================================================================
# 2. VÉRIFICATIONS DES RÉSULTATS PRÉCÉDENTS
# ==============================================================================

assert (
    "child_frame_summary_fallback_df"
    in globals()
), (
    "Le tableau des détections enfant est absent."
)

assert (
    "object_detections_df"
    in globals()
), (
    "Le tableau des détections YOLOE-26s est absent."
)

assert (
    "extracted_frames_df"
    in globals()
), (
    "L'index des frames extraites est absent."
)

assert len(
    child_frame_summary_fallback_df
) == EXPECTED_NUMBER_OF_FRAMES, (
    f"{EXPECTED_NUMBER_OF_FRAMES} résultats de détection enfant sont attendus."
)

assert not object_detections_df.empty, (
    "Le tableau des détections YOLOE-26s est vide."
)


# ==============================================================================
# 3. COLONNES UTILISÉES POUR LA FUSION
# ==============================================================================

merge_keys = [
    "video_id",
    "sample_index",
    "frame_index",
    "frame_path",
]

child_columns = (
    merge_keys
    + [
        "child_detected",
        "child_confidence",
        "child_bbox_x1",
        "child_bbox_y1",
        "child_bbox_x2",
        "child_bbox_y2",
        "child_detection_source",
    ]
)

missing_child_columns = [
    column
    for column in child_columns
    if column
    not in child_frame_summary_fallback_df.columns
]

assert not missing_child_columns, (
    "Colonnes enfant manquantes : "
    f"{missing_child_columns}"
)

required_object_columns = [
    "video_id",
    "true_activity_class",
    "sample_index",
    "frame_index",
    "timestamp_seconds",
    "frame_path",
    "detected_class_name",
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
]

missing_object_columns = [
    column
    for column in required_object_columns
    if column
    not in object_detections_df.columns
]

assert not missing_object_columns, (
    "Colonnes objets manquantes : "
    f"{missing_object_columns}"
)


# ==============================================================================
# 4. RÉCUPÉRATION DES DIMENSIONS DES FRAMES
# ==============================================================================

required_frame_columns = [
    "frame_path",
    "width",
    "height",
]

missing_frame_columns = [
    column
    for column in required_frame_columns
    if column
    not in extracted_frames_df.columns
]

assert not missing_frame_columns, (
    "Colonnes de dimensions manquantes : "
    f"{missing_frame_columns}"
)

frame_dimensions_df = (
    extracted_frames_df[
        required_frame_columns
    ]
    .drop_duplicates(
        subset=["frame_path"]
    )
    .copy()
)

frame_dimensions_df[
    "width"
] = pd.to_numeric(
    frame_dimensions_df["width"],
    errors="coerce",
)

frame_dimensions_df[
    "height"
] = pd.to_numeric(
    frame_dimensions_df["height"],
    errors="coerce",
)

assert frame_dimensions_df[
    ["width", "height"]
].notna().all().all(), (
    "Certaines dimensions sont manquantes."
)

assert (
    frame_dimensions_df["width"] > 0
).all(), (
    "Certaines largeurs sont invalides."
)

assert (
    frame_dimensions_df["height"] > 0
).all(), (
    "Certaines hauteurs sont invalides."
)


# ==============================================================================
# 5. FUSION OBJETS + ENFANT + DIMENSIONS
# ==============================================================================

# Évite la création de width_x, width_y, height_x et height_y
# si cette cellule est exécutée plusieurs fois.

objects_for_merge_df = (
    object_detections_df
    .drop(
        columns=[
            "width",
            "height",
        ],
        errors="ignore",
    )
    .copy()
)

child_for_merge_df = (
    child_frame_summary_fallback_df[
        child_columns
    ]
    .copy()
)

child_object_context_df = (
    objects_for_merge_df
    .merge(
        child_for_merge_df,
        on=merge_keys,
        how="left",
        validate="many_to_one",
    )
    .merge(
        frame_dimensions_df,
        on="frame_path",
        how="left",
        validate="many_to_one",
    )
)

assert len(
    child_object_context_df
) == len(
    object_detections_df
), (
    "La fusion a modifié le nombre "
    "de détections d'objets."
)

assert child_object_context_df[
    ["width", "height"]
].notna().all().all(), (
    "Certaines détections n'ont pas "
    "de dimensions d'image."
)


# ==============================================================================
# 6. PRÉPARATION DES DIMENSIONS
# ==============================================================================

width = child_object_context_df[
    "width"
].astype(float)

height = child_object_context_df[
    "height"
].astype(float)

diagonal = np.hypot(
    width,
    height,
)

assert (
    diagonal > 0
).all(), (
    "Certaines diagonales d'image sont invalides."
)


# ==============================================================================
# 7. NORMALISATION DES BOUNDING BOXES DES OBJETS
# ==============================================================================

object_normalization = {
    "bbox_x1": width,
    "bbox_x2": width,
    "bbox_y1": height,
    "bbox_y2": height,
}

for coordinate, denominator in (
    object_normalization.items()
):

    normalized_column = (
        f"object_{coordinate}_normalized"
    )

    child_object_context_df[
        normalized_column
    ] = (
        child_object_context_df[
            coordinate
        ].astype(float)
        / denominator
    ).clip(
        0.0,
        1.0,
    )


# ==============================================================================
# 8. CALCUL DU CENTRE DES OBJETS
# ==============================================================================

child_object_context_df[
    "object_center_x"
] = (
    child_object_context_df["bbox_x1"]
    + child_object_context_df["bbox_x2"]
) / 2.0

child_object_context_df[
    "object_center_y"
] = (
    child_object_context_df["bbox_y1"]
    + child_object_context_df["bbox_y2"]
) / 2.0

child_object_context_df[
    "object_center_x_normalized"
] = (
    child_object_context_df[
        "object_center_x"
    ]
    / width
).clip(
    0.0,
    1.0,
)

child_object_context_df[
    "object_center_y_normalized"
] = (
    child_object_context_df[
        "object_center_y"
    ]
    / height
).clip(
    0.0,
    1.0,
)


# ==============================================================================
# 9. IDENTIFICATION DES FRAMES OÙ L'ENFANT EST DISPONIBLE
# ==============================================================================

child_bbox_columns = [
    "child_bbox_x1",
    "child_bbox_y1",
    "child_bbox_x2",
    "child_bbox_y2",
]

child_available = (
    child_object_context_df[
        "child_detected"
    ]
    .fillna(False)
    .astype(bool)
    &
    child_object_context_df[
        child_bbox_columns
    ]
    .notna()
    .all(axis=1)
)

child_object_context_df[
    "child_available_for_geometry"
] = child_available


# ==============================================================================
# 10. NORMALISATION DE LA BOUNDING BOX DE L'ENFANT
# ==============================================================================

child_normalization = {
    "child_bbox_x1": width,
    "child_bbox_x2": width,
    "child_bbox_y1": height,
    "child_bbox_y2": height,
}

for coordinate, denominator in (
    child_normalization.items()
):

    normalized_column = (
        f"{coordinate}_normalized"
    )

    child_object_context_df[
        normalized_column
    ] = np.nan

    child_object_context_df.loc[
        child_available,
        normalized_column,
    ] = (
        child_object_context_df.loc[
            child_available,
            coordinate,
        ].astype(float)
        / denominator[
            child_available
        ]
    ).clip(
        0.0,
        1.0,
    )


# ==============================================================================
# 11. CALCUL DU CENTRE DE L'ENFANT
# ==============================================================================

child_object_context_df[
    "child_center_x"
] = np.nan

child_object_context_df[
    "child_center_y"
] = np.nan

child_object_context_df.loc[
    child_available,
    "child_center_x",
] = (
    child_object_context_df.loc[
        child_available,
        "child_bbox_x1",
    ]
    +
    child_object_context_df.loc[
        child_available,
        "child_bbox_x2",
    ]
) / 2.0

child_object_context_df.loc[
    child_available,
    "child_center_y",
] = (
    child_object_context_df.loc[
        child_available,
        "child_bbox_y1",
    ]
    +
    child_object_context_df.loc[
        child_available,
        "child_bbox_y2",
    ]
) / 2.0

child_object_context_df[
    "child_center_x_normalized"
] = (
    child_object_context_df[
        "child_center_x"
    ]
    / width
)

child_object_context_df[
    "child_center_y_normalized"
] = (
    child_object_context_df[
        "child_center_y"
    ]
    / height
)


# ==============================================================================
# 12. DISTANCE ENTRE LES CENTRES
# ==============================================================================

child_object_context_df[
    "center_distance_pixels"
] = np.nan

child_object_context_df.loc[
    child_available,
    "center_distance_pixels",
] = np.hypot(
    (
        child_object_context_df.loc[
            child_available,
            "object_center_x",
        ]
        -
        child_object_context_df.loc[
            child_available,
            "child_center_x",
        ]
    ),
    (
        child_object_context_df.loc[
            child_available,
            "object_center_y",
        ]
        -
        child_object_context_df.loc[
            child_available,
            "child_center_y",
        ]
    ),
)

# Normalisation par la diagonale de l'image.

child_object_context_df[
    "center_distance_normalized"
] = (
    child_object_context_df[
        "center_distance_pixels"
    ]
    / diagonal
)


# ==============================================================================
# 13. DISTANCE MINIMALE ENTRE LES DEUX RECTANGLES
# ==============================================================================

# Cette distance vaut zéro lorsque les deux bounding boxes
# se touchent ou se chevauchent.

horizontal_gap = np.maximum.reduce(
    [
        (
            child_object_context_df[
                "child_bbox_x1"
            ].to_numpy()
            -
            child_object_context_df[
                "bbox_x2"
            ].to_numpy()
        ),
        (
            child_object_context_df[
                "bbox_x1"
            ].to_numpy()
            -
            child_object_context_df[
                "child_bbox_x2"
            ].to_numpy()
        ),
        np.zeros(
            len(child_object_context_df)
        ),
    ]
)

vertical_gap = np.maximum.reduce(
    [
        (
            child_object_context_df[
                "child_bbox_y1"
            ].to_numpy()
            -
            child_object_context_df[
                "bbox_y2"
            ].to_numpy()
        ),
        (
            child_object_context_df[
                "bbox_y1"
            ].to_numpy()
            -
            child_object_context_df[
                "child_bbox_y2"
            ].to_numpy()
        ),
        np.zeros(
            len(child_object_context_df)
        ),
    ]
)

child_object_context_df[
    "bbox_gap_pixels"
] = np.nan

child_object_context_df.loc[
    child_available,
    "bbox_gap_pixels",
] = np.hypot(
    horizontal_gap[
        child_available.to_numpy()
    ],
    vertical_gap[
        child_available.to_numpy()
    ],
)

child_object_context_df[
    "bbox_gap_normalized"
] = (
    child_object_context_df[
        "bbox_gap_pixels"
    ]
    / diagonal
)


# ==============================================================================
# 14. CRÉATION DES CLÉS DE FRAME
# ==============================================================================

child_object_context_df[
    "frame_key"
] = (
    child_object_context_df[
        "video_id"
    ].astype(str)
    + "__"
    + child_object_context_df[
        "sample_index"
    ].astype(str)
)

child_object_context_df[
    "frame_key_with_child"
] = (
    child_object_context_df[
        "frame_key"
    ]
    .where(child_available)
)


# ==============================================================================
# 15. TRI DES DÉTECTIONS
# ==============================================================================

child_object_context_df = (
    child_object_context_df
    .sort_values(
        [
            "video_id",
            "sample_index",
            "center_distance_normalized",
            "confidence",
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


# ==============================================================================
# 16. RÉSUMÉ TEMPOREL PAR OBJET
# ==============================================================================

object_temporal_summary_df = (
    child_object_context_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
            "detected_class_name",
        ],
        as_index=False,
    )
    .agg(
        detection_count=(
            "detected_class_name",
            "size",
        ),
        frames_present=(
            "frame_key",
            "nunique",
        ),
        first_timestamp_seconds=(
            "timestamp_seconds",
            "min",
        ),
        last_timestamp_seconds=(
            "timestamp_seconds",
            "max",
        ),
        mean_confidence=(
            "confidence",
            "mean",
        ),
        maximum_confidence=(
            "confidence",
            "max",
        ),
        minimum_center_distance_normalized=(
            "center_distance_normalized",
            "min",
        ),
        mean_center_distance_normalized=(
            "center_distance_normalized",
            "mean",
        ),
        minimum_bbox_gap_normalized=(
            "bbox_gap_normalized",
            "min",
        ),
    )
)


# ==============================================================================
# 17. ESTIMATION DE LA DURÉE DE PRÉSENCE
# ==============================================================================

duration_lookup = (
    selected_videos_df[
        [
            "video_id",
            "duration_seconds",
        ]
    ]
    .rename(
        columns={
            "duration_seconds": "video_duration_seconds",
        }
    )
)

object_temporal_summary_df = (
    object_temporal_summary_df
    .merge(
        duration_lookup,
        on="video_id",
        how="left",
        validate="many_to_one",
    )
)

object_temporal_summary_df[
    "presence_ratio"
] = (
    object_temporal_summary_df[
        "frames_present"
    ]
    / NUM_CONTEXT_FRAMES
)

object_temporal_summary_df[
    "estimated_presence_seconds"
] = (
    object_temporal_summary_df[
        "presence_ratio"
    ]
    *
    object_temporal_summary_df[
        "video_duration_seconds"
    ]
)


# ==============================================================================
# 18. RÉSUMÉ DU CONTEXTE PAR VIDÉO
# ==============================================================================

video_context_summary_df = (
    child_object_context_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
        ],
        as_index=False,
    )
    .agg(
        total_object_detections=(
            "detected_class_name",
            "size",
        ),
        object_classes_detected=(
            "detected_class_name",
            "nunique",
        ),
        frames_with_objects=(
            "frame_key",
            "nunique",
        ),
        frames_with_child_and_objects=(
            "frame_key_with_child",
            "nunique",
        ),
        minimum_center_distance_normalized=(
            "center_distance_normalized",
            "min",
        ),
    )
)


# ==============================================================================
# 19. SAUVEGARDE DES RÉSULTATS
# ==============================================================================

child_object_context_df.to_csv(
    CHILD_OBJECT_GEOMETRY_CSV_PATH,
    index=False,
)

object_temporal_summary_df.to_csv(
    OBJECT_TEMPORAL_SUMMARY_CSV_PATH,
    index=False,
)

video_context_summary_df.to_csv(
    VIDEO_CONTEXT_SUMMARY_CSV_PATH,
    index=False,
)


# ==============================================================================
# 20. TESTS DE COHÉRENCE
# ==============================================================================

assert (
    child_object_context_df[
        "center_distance_normalized"
    ]
    .dropna()
    .ge(0)
    .all()
), (
    "Certaines distances normalisées "
    "sont négatives."
)

assert (
    object_temporal_summary_df[
        "presence_ratio"
    ]
    .between(
        0,
        1,
    )
    .all()
), (
    "Certains ratios de présence "
    "sont invalides."
)

assert len(
    child_object_context_df
) == len(
    object_detections_df
)


# ==============================================================================
# 21. AFFICHAGE DES RÉSULTATS
# ==============================================================================

print("=" * 80)
print("FUSION DESCRIPTIVE TERMINÉE")
print("=" * 80)

print(
    "Détections d'objets :",
    len(child_object_context_df),
)

print(
    "Distances calculables :",
    int(child_available.sum()),
)

print(
    "Objets temporels résumés :",
    len(object_temporal_summary_df),
)

print(
    "Aucune décision near/far "
    "n'est prise dans le Module 3."
)

print(
    "Dimensions utilisées :",
    child_object_context_df[
        ["width", "height"]
    ]
    .drop_duplicates()
    .to_dict("records"),
)

print(
    "CSV géométrie :",
    CHILD_OBJECT_GEOMETRY_CSV_PATH,
)

print(
    "CSV temporel :",
    OBJECT_TEMPORAL_SUMMARY_CSV_PATH,
)

print(
    "CSV résumé vidéo :",
    VIDEO_CONTEXT_SUMMARY_CSV_PATH,
)

child_object_context_df.to_csv(
    MODULE_OUTPUTS_DIR / "child_object_geometry.csv",
    index=False,
)
object_temporal_summary_df.to_csv(
    MODULE_OUTPUTS_DIR / "object_temporal_summary.csv",
    index=False,
)

print("\nCELLULE 8 TERMINÉE")


In [ ]:
# ==============================================================================
# CELLULE 8 — JSON FINAL PAR VIDÉO
# Sortie factuelle destinée au module LLM, sans décision de risque
# ==============================================================================

from pathlib import Path
import json
import math
import shutil
import time
import numpy as np
import pandas as pd


JSON_FULL_DIR = PROJECT_DIR / "json_outputs_final"
if JSON_FULL_DIR.exists():
    shutil.rmtree(JSON_FULL_DIR)
JSON_FULL_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_JSON_FULL_PATH = JSON_FULL_DIR / "context_all_videos_final.json"


def to_float(value, digits=6):
    if value is None or pd.isna(value):
        return None
    return round(float(value), digits)


def to_int(value):
    if value is None or pd.isna(value):
        return None
    return int(value)


def bbox_values(row, prefix=""):
    keys = [f"{prefix}bbox_x1", f"{prefix}bbox_y1", f"{prefix}bbox_x2", f"{prefix}bbox_y2"]
    if any(pd.isna(row.get(key)) for key in keys):
        return None
    return [round(float(row[key]), 2) for key in keys]


def normalized_bbox_values(row, prefix=""):
    if prefix == "child_":
        keys = [
            "child_bbox_x1_normalized", "child_bbox_y1_normalized",
            "child_bbox_x2_normalized", "child_bbox_y2_normalized",
        ]
    else:
        keys = [
            "object_bbox_x1_normalized", "object_bbox_y1_normalized",
            "object_bbox_x2_normalized", "object_bbox_y2_normalized",
        ]
    if any(pd.isna(row.get(key)) for key in keys):
        return None
    return [round(float(row[key]), 6) for key in keys]


def object_detection_to_json(row, rank):
    return {
        "distance_rank": int(rank),
        "name": str(row["detected_class_name"]),
        "confidence": to_float(row["confidence"]),
        "bbox_pixels": bbox_values(row),
        "bbox_normalized": normalized_bbox_values(row),
        "center_normalized": [
            to_float(row["object_center_x_normalized"]),
            to_float(row["object_center_y_normalized"]),
        ],
        "distance_to_child": {
            "available": bool(row["child_available_for_geometry"]),
            "center_distance_pixels": to_float(row["center_distance_pixels"], 2),
            "center_distance_normalized": to_float(row["center_distance_normalized"]),
            "bbox_gap_pixels": to_float(row["bbox_gap_pixels"], 2),
            "bbox_gap_normalized": to_float(row["bbox_gap_normalized"]),
        },
    }


def temporal_object_to_json(row):
    return {
        "name": str(row["detected_class_name"]),
        "detection_count": int(row["detection_count"]),
        "frames_present": int(row["frames_present"]),
        "presence_ratio": to_float(row["presence_ratio"], 4),
        "estimated_presence_seconds": to_float(row["estimated_presence_seconds"], 3),
        "first_timestamp_seconds": to_float(row["first_timestamp_seconds"], 3),
        "last_timestamp_seconds": to_float(row["last_timestamp_seconds"], 3),
        "mean_confidence": to_float(row["mean_confidence"]),
        "maximum_confidence": to_float(row["maximum_confidence"]),
        "minimum_center_distance_normalized": to_float(
            row["minimum_center_distance_normalized"]
        ),
        "mean_center_distance_normalized": to_float(
            row["mean_center_distance_normalized"]
        ),
        "minimum_bbox_gap_normalized": to_float(
            row["minimum_bbox_gap_normalized"]
        ),
    }


assert len(videomae_predictions_df) == EXPECTED_NUMBER_OF_VIDEOS
assert len(child_frame_summary_fallback_df) == EXPECTED_NUMBER_OF_FRAMES
assert len(child_object_context_df) == len(object_detections_df)

activity_lookup = videomae_predictions_df.set_index("video_id").to_dict("index")
yoloe_time_by_video = yoloe_latency_df.groupby("video_id")["latency_seconds"].sum().to_dict()

frame_time_per_video = float(FRAME_EXTRACTION_TOTAL_SECONDS) / float(EXPECTED_NUMBER_OF_VIDEOS)
child_time_per_video = float(CHILD_INFERENCE_TOTAL_SECONDS) / float(EXPECTED_NUMBER_OF_VIDEOS)

json_generation_start = time.perf_counter()
all_video_json_outputs = []

for video_row in selected_videos_df.itertuples(index=False):
    video_id = str(video_row.video_id)
    matching_frames = extracted_frames_df[
        extracted_frames_df["video_id"].astype(str) == video_id
    ]
    assert len(matching_frames) == NUM_CONTEXT_FRAMES
    assert video_id in activity_lookup

    activity = activity_lookup[video_id]
    child_frames = child_frame_summary_fallback_df[
        child_frame_summary_fallback_df["video_id"].astype(str) == video_id
    ].merge(
        extracted_frames_df[
            ["video_id", "sample_index", "frame_index", "frame_path", "width", "height"]
        ],
        on=["video_id", "sample_index", "frame_index", "frame_path"],
        how="left",
        validate="one_to_one",
    ).sort_values("sample_index")
    video_objects = child_object_context_df[
        child_object_context_df["video_id"].astype(str) == video_id
    ]
    temporal_objects = object_temporal_summary_df[
        object_temporal_summary_df["video_id"].astype(str) == video_id
    ].sort_values(
        ["minimum_center_distance_normalized", "frames_present"],
        ascending=[True, False],
        na_position="last",
    )

    frames_json = []
    for child_row in child_frames.itertuples(index=False):
        child_series = pd.Series(child_row._asdict())
        frame_objects = video_objects[
            video_objects["sample_index"] == int(child_row.sample_index)
        ].sort_values(
            ["center_distance_normalized", "confidence"],
            ascending=[True, False],
            na_position="last",
        )
        objects_json = [
            object_detection_to_json(object_row, rank)
            for rank, (_, object_row) in enumerate(frame_objects.iterrows(), start=1)
        ]

        child_detected = bool(child_row.child_detected)
        child_bbox_normalized = None
        child_center_normalized = None
        if child_detected:
            width_value = float(child_row.width)
            height_value = float(child_row.height)
            child_bbox_normalized = [
                to_float(child_row.child_bbox_x1 / width_value),
                to_float(child_row.child_bbox_y1 / height_value),
                to_float(child_row.child_bbox_x2 / width_value),
                to_float(child_row.child_bbox_y2 / height_value),
            ]
            child_center_normalized = [
                to_float((child_row.child_bbox_x1 + child_row.child_bbox_x2) / (2 * width_value)),
                to_float((child_row.child_bbox_y1 + child_row.child_bbox_y2) / (2 * height_value)),
            ]

        frames_json.append({
            "sample_index": int(child_row.sample_index),
            "frame_index": int(child_row.frame_index),
            "timestamp_seconds": to_float(child_row.timestamp_seconds, 3),
            "image_size": {"width": int(child_row.width), "height": int(child_row.height)},
            "child": {
                "detected": child_detected,
                "confidence": to_float(child_row.child_confidence),
                "detection_source": str(child_row.child_detection_source),
                "bbox_pixels": bbox_values(child_series, prefix="child_"),
                "bbox_normalized": child_bbox_normalized,
                "center_normalized": child_center_normalized,
            },
            "number_of_objects": len(objects_json),
            "objects_sorted_by_distance": objects_json,
        })

    video_json = {
        "video_id": video_id,
        "split": str(video_row.split),
        "reference_activity_class": str(video_row.class_name),
        "video_duration_seconds": to_float(video_row.duration_seconds, 3),
        "activity": {
            "predicted_class": str(activity["predicted_class"]),
            "confidence": to_float(activity["confidence"]),
            "is_abnormal": bool(activity["is_abnormal"]),
            "source": "videomae_checkpoint",
        },
        "child_detection": {
            "frames_analyzed": len(child_frames),
            "frames_detected": int(child_frames["child_detected"].sum()),
            "detection_rate": to_float(child_frames["child_detected"].mean(), 4),
            "fallback_detections": int(
                child_frames["child_detection_source"].eq("fallback_threshold_0.10").sum()
            ),
        },
        "context": {
            "number_of_object_detections": len(video_objects),
            "number_of_object_classes": int(video_objects["detected_class_name"].nunique()),
            "temporal_object_summary": [
                temporal_object_to_json(row)
                for _, row in temporal_objects.iterrows()
            ],
            "geometry_policy": (
                "continuous normalized geometry only; no near/far or risk decision"
            ),
        },
        "frames": frames_json,
    }
    all_video_json_outputs.append(video_json)

JSON_GENERATION_TOTAL_SECONDS = time.perf_counter() - json_generation_start
json_time_per_video = JSON_GENERATION_TOTAL_SECONDS / float(EXPECTED_NUMBER_OF_VIDEOS)

for video_json in all_video_json_outputs:
    video_id = video_json["video_id"]
    activity_seconds = float(activity_lookup[video_id]["mean_processing_time_per_video_seconds"])
    object_seconds = float(yoloe_time_by_video.get(video_id, 0.0))
    stage_times = {
        "frame_extraction": frame_time_per_video,
        "activity_inference_videomae": activity_seconds,
        "child_inference_yolo26m": child_time_per_video,
        "object_inference_yoloe_26s": object_seconds,
        "fusion_and_json_generation": json_time_per_video,
    }
    video_json["processing_time_seconds"] = {
        **{name: round(value, 4) for name, value in stage_times.items()},
        "total": round(sum(stage_times.values()), 4),
        "timing_scope": {
            "videomae": "subprocess startup, checkpoint loading and inference",
            "yolo26m": "GPU inference only, including fallback when used",
            "yoloe_26s": "GPU inference only after warm-up",
        },
        "excludes": [
            "dependency installation",
            "model download",
            "benchmark calibration",
            "visualization generation",
        ],
    }

for video_json in all_video_json_outputs:
    safe_video_id = video_json["video_id"].replace("/", "_").replace("\\", "_")
    with (JSON_FULL_DIR / f"context_{safe_video_id}_final.json").open(
        "w", encoding="utf-8"
    ) as json_file:
        json.dump(video_json, json_file, indent=2, ensure_ascii=False)

combined_json_full = {
    "module": "P3_context_extraction",
    "number_of_videos": len(all_video_json_outputs),
    "number_of_frames_per_video": NUM_CONTEXT_FRAMES,
    "total_object_detections": len(child_object_context_df),
    "decision_policy": (
        "descriptive output only; the following LLM module performs risk assessment"
    ),
    "activity_detector": {
        "model": "VideoMAE",
        "checkpoint": "original_training_checkpoint",
        "number_of_classes": 12,
        "output_policy": "raw prediction and confidence; no activity decision threshold",
    },
    "child_detector": {"model": "YOLO26m", "primary_threshold": 0.25, "fallback_threshold": 0.10},
    "object_detector": {
        "model": "YOLOE-26s",
        "checkpoint": YOLOE_MODEL_ID,
        "confidence_threshold": float(OBJECT_CONFIDENCE_THRESHOLD),
        "nms_iou_threshold": float(YOLOE_NMS_IOU),
        "number_of_prompt_classes": len(YOLOE_CONTEXT_CLASSES),
    },
    "videos": all_video_json_outputs,
}

with COMBINED_JSON_FULL_PATH.open("w", encoding="utf-8") as json_file:
    json.dump(combined_json_full, json_file, indent=2, ensure_ascii=False)

json_detection_count = sum(
    video["context"]["number_of_object_detections"]
    for video in all_video_json_outputs
)
assert json_detection_count == len(child_object_context_df)

print("=" * 80)
print("JSON FINAL CRÉÉ")
print("=" * 80)
print("JSON vidéo :", len(all_video_json_outputs))
print("Détections enregistrées :", json_detection_count)
print("Fichier combiné :", COMBINED_JSON_FULL_PATH)
print("Décision near/far : absente")

print("\nCELLULE 9 TERMINÉE")


## Décision finale gelée — Mistral few-shot

Cette cellule ne sélectionne plus de modèle et ne calibre plus rien. Elle recharge **uniquement la configuration retenue** et applique exactement le mode few-shot gelé.

Elle recherche automatiquement `calibration_inputs.json` et `ground_truth_calibration_annotated.csv` dans `/kaggle/input` (ou dans `/kaggle/working` si le test est exécuté dans la même session).


In [ ]:
# ============================================================
# CELLULE 10 — MISTRAL FEW-SHOT GELÉ + TEST SYSTÈME FINAL
# ============================================================

%pip install -q -U "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.44"

import gc
import json
import re
import time
from pathlib import Path

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ------------------------------------------------------------
# 1. Retrouver les deux références gelées de calibration
# ------------------------------------------------------------

SEARCH_REFERENCE_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

def find_unique_file(filename):
    matches = []
    for root in SEARCH_REFERENCE_ROOTS:
        if root.exists():
            matches.extend(
                p for p in root.rglob(filename)
                if p.is_file()
            )

    # Priorité aux fichiers sous /kaggle/input puis aux fichiers de working
    matches = sorted(
        set(matches),
        key=lambda p: (
            0 if str(p).startswith("/kaggle/input") else 1,
            len(p.parts),
        ),
    )

    print(f"\n{filename} :")
    for p in matches:
        print(" -", p)

    assert matches, (
        f"{filename} introuvable. "
        "Ajoute ce fichier comme Input Kaggle."
    )

    return matches[0]

CALIBRATION_INPUTS_PATH = find_unique_file("calibration_inputs.json")
CALIBRATION_GT_PATH = find_unique_file("ground_truth_calibration_annotated.csv")

with CALIBRATION_INPUTS_PATH.open("r", encoding="utf-8") as f:
    calibration_decision_inputs = json.load(f)

calibration_annotation_df = pd.read_csv(CALIBRATION_GT_PATH)

assert len(calibration_decision_inputs) == 24
assert len(calibration_annotation_df) == 24

calibration_labels = {
    str(row.video_id): str(row.expected_hazard).strip()
    for row in calibration_annotation_df.itertuples(index=False)
}

assert set(calibration_labels.values()) <= {"hazard", "no_hazard"}

# ------------------------------------------------------------
# 2. Entrées E2E produites par toute la chaîne
# ------------------------------------------------------------

e2e_decision_inputs = all_video_json_outputs

assert len(e2e_decision_inputs) == EXPECTED_NUMBER_OF_VIDEOS == 5

# ------------------------------------------------------------
# 3. Prompt EXACT de la configuration finale
# ------------------------------------------------------------

SYSTEM_PROMPT = """You are the final decision module of a child-safety monitoring system.

You receive factual outputs from upstream computer-vision models:
- child detection,
- recognized activity and confidence,
- objects detected in the scene,
- temporal persistence,
- child-object distances.

Decide whether the observed situation is hazardous for the child.

Important:
- An abnormal activity is NOT automatically a hazard.
- The mere presence of an object is NOT automatically a hazard.
- Use activity, context, persistence and proximity together.
- Do not invent information that is absent from the structured input.

Return ONLY valid JSON with exactly these keys:
{"hazard":"hazard" or "no_hazard","reason":"short explanation"}
"""

def compact_decision_payload(item, max_objects=12):
    context = item.get("context", {})

    objects = context.get(
        "temporal_objects",
        context.get("temporal_object_summary", []),
    )

    compact_objects = []

    for obj in objects[:max_objects]:
        compact_objects.append({
            "name": obj.get("name"),
            "presence_ratio": obj.get("presence_ratio"),
            "minimum_center_distance_normalized":
                obj.get("minimum_center_distance_normalized"),
            "minimum_bbox_gap_normalized":
                obj.get("minimum_bbox_gap_normalized"),
            "maximum_confidence":
                obj.get("maximum_confidence", obj.get("confidence_max")),
        })

    return {
        "video_id": item.get("video_id"),
        "child": item.get(
            "child",
            item.get("child_detection", {}),
        ),
        "activity": item.get("activity", {}),
        "context": {
            "number_of_object_classes":
                context.get("number_of_object_classes"),
            "number_of_object_detections":
                context.get("number_of_object_detections"),
            "minimum_center_distance_normalized":
                context.get("minimum_center_distance_normalized"),
            "minimum_bbox_gap_normalized":
                context.get("minimum_bbox_gap_normalized"),
            "objects": compact_objects,
        },
    }

def build_user_prompt(item):
    return (
        "Analyze this structured child-safety observation and decide the risk.\n\n"
        + json.dumps(
            compact_decision_payload(item),
            ensure_ascii=False,
            separators=(",", ":"),
        )
    )

# ------------------------------------------------------------
# 4. Reconstruire les 4 exemples few-shot gelés
#    Même politique que la Partie 4 : 2 hazard + 2 no_hazard
# ------------------------------------------------------------

hazard_examples = []
safe_examples = []

for item in calibration_decision_inputs:
    vid = str(item["video_id"])
    label = calibration_labels[vid]

    if label == "hazard":
        hazard_examples.append((item, label))
    else:
        safe_examples.append((item, label))

FEW_SHOT_EXAMPLES = hazard_examples[:2] + safe_examples[:2]

assert len(FEW_SHOT_EXAMPLES) == 4
assert sum(label == "hazard" for _, label in FEW_SHOT_EXAMPLES) == 2
assert sum(label == "no_hazard" for _, label in FEW_SHOT_EXAMPLES) == 2

print("\nExemples few-shot gelés :")
for item, label in FEW_SHOT_EXAMPLES:
    print(" -", item["video_id"], "=>", label)

def build_messages(item):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        }
    ]

    for example_item, example_label in FEW_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": build_user_prompt(example_item),
        })

        messages.append({
            "role": "assistant",
            "content": json.dumps({
                "hazard": example_label,
                "reason": "reference calibration example",
            }),
        })

    messages.append({
        "role": "user",
        "content": build_user_prompt(item),
    })

    return messages

# ------------------------------------------------------------
# 5. Parser JSON STRICT
# ------------------------------------------------------------

def parse_final_json(raw_text):
    result = {
        "hazard": None,
        "reason": None,
        "valid_json": False,
    }

    text = str(raw_text).strip()

    if not text:
        return result

    text = (
        text
        .replace("```json", "")
        .replace("```JSON", "")
        .replace("```", "")
        .strip()
    )

    candidates = [text]
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)

    if match:
        candidate = match.group(0).strip()
        if candidate not in candidates:
            candidates.append(candidate)

    for candidate in candidates:
        try:
            obj = json.loads(candidate)
        except Exception:
            continue

        if not isinstance(obj, dict):
            continue

        hazard = str(obj.get("hazard", "")).strip()
        reason = obj.get("reason", "")

        if (
            hazard in {"hazard", "no_hazard"}
            and isinstance(reason, str)
        ):
            return {
                "hazard": hazard,
                "reason": reason.strip(),
                "valid_json": True,
            }

    return result

# ------------------------------------------------------------
# 6. Charger UNIQUEMENT le Mistral final
# ------------------------------------------------------------

FINAL_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_ID,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    FINAL_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model.eval()

device = model.get_input_embeddings().weight.device

# ------------------------------------------------------------
# 7. Inférence finale sur les 5 vidéos
# ------------------------------------------------------------

decision_records = []

for i, item in enumerate(e2e_decision_inputs, start=1):
    video_id = str(item["video_id"])

    messages = build_messages(item)

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            use_cache=True,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decision_latency = time.perf_counter() - start

    generated = outputs[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    raw_output = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    parsed = parse_final_json(raw_output)

    expected = E2E_EXPECTED[video_id]

    processing_time = item.get(
        "processing_time_seconds", {}
    )

    upstream_seconds = float(
        processing_time.get("total", 0.0)
    )

    end_to_end_seconds = (
        upstream_seconds
        + float(decision_latency)
    )

    correct = (
        parsed["hazard"] == expected
        if parsed["valid_json"]
        else False
    )

    decision_records.append({
        "video_id": video_id,
        "reference_activity":
            item.get("reference_activity_class"),
        "videomae_prediction":
            item.get("activity", {}).get("predicted_class"),
        "videomae_confidence":
            item.get("activity", {}).get("confidence"),
        "child_detection_rate":
            item.get("child_detection", {}).get("detection_rate"),
        "object_classes":
            item.get("context", {}).get("number_of_object_classes"),
        "object_detections":
            item.get("context", {}).get("number_of_object_detections"),
        "expected_hazard": expected,
        "predicted_hazard": parsed["hazard"],
        "valid_json": parsed["valid_json"],
        "correct": bool(correct),
        "reason": parsed["reason"],
        "upstream_seconds": upstream_seconds,
        "decision_seconds": float(decision_latency),
        "end_to_end_seconds": float(end_to_end_seconds),
        "raw_output": raw_output,
    })

    print(
        f"[{i:02d}/05] {video_id} -> "
        f"{parsed['hazard']} | GT={expected} | "
        f"correct={correct} | JSON={parsed['valid_json']}"
    )

e2e_results_df = pd.DataFrame(decision_records)

# ------------------------------------------------------------
# 8. Tests système explicites
# ------------------------------------------------------------

system_tests = []

def system_test(name, condition, details=""):
    passed = bool(condition)
    system_tests.append({
        "test": name,
        "passed": passed,
        "details": str(details),
    })
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name} {details}")
    return passed

print("\n" + "=" * 100)
print("TESTS SYSTÈME END-TO-END")
print("=" * 100)

system_test(
    "5 vidéos traitées",
    len(e2e_results_df) == 5,
    f"n={len(e2e_results_df)}",
)

system_test(
    "16 frames par vidéo",
    extracted_frames_df.groupby("video_id").size().eq(16).all(),
)

system_test(
    "YOLO26m produit un résumé pour chaque frame",
    len(child_frame_summary_fallback_df) == EXPECTED_NUMBER_OF_FRAMES,
    f"n={len(child_frame_summary_fallback_df)}",
)

system_test(
    "VideoMAE produit une prédiction par vidéo",
    len(videomae_predictions_df) == 5,
    f"n={len(videomae_predictions_df)}",
)

system_test(
    "JSON factuel produit par vidéo",
    len(all_video_json_outputs) == 5,
)

system_test(
    "Mistral produit une décision par vidéo",
    e2e_results_df["predicted_hazard"].notna().all(),
)

system_test(
    "100% JSON valides",
    e2e_results_df["valid_json"].all(),
    f"{e2e_results_df['valid_json'].mean()*100:.1f}%",
)

system_test(
    "Aucune fuite de vérité terrain dans le JSON factuel",
    all(
        "expected_hazard" not in json.dumps(item, ensure_ascii=False)
        for item in all_video_json_outputs
    ),
)

system_test(
    "Toutes les étapes ont une sortie exploitable",
    (
        len(extracted_frames_df) == 80
        and len(child_frame_summary_fallback_df) == 80
        and len(videomae_predictions_df) == 5
        and len(all_video_json_outputs) == 5
        and len(e2e_results_df) == 5
    ),
)

system_tests_df = pd.DataFrame(system_tests)

# ------------------------------------------------------------
# 9. Résumé système
# ------------------------------------------------------------

accuracy = float(e2e_results_df["correct"].mean())
valid_json_rate = float(e2e_results_df["valid_json"].mean())
mean_e2e_latency = float(e2e_results_df["end_to_end_seconds"].mean())

print("\n" + "=" * 100)
print("RÉSULTAT DU TEST END-TO-END")
print("=" * 100)

print("Vidéos                  :", len(e2e_results_df))
print("Décisions correctes     :", int(e2e_results_df["correct"].sum()), "/ 5")
print("Accuracy sur échantillon:", round(accuracy, 4))
print("JSON valides            :", round(valid_json_rate, 4))
print("Latence E2E moyenne     :", round(mean_e2e_latency, 3), "s")
print("Tests système PASS      :", int(system_tests_df["passed"].sum()), "/", len(system_tests_df))

display(
    e2e_results_df[
        [
            "video_id",
            "reference_activity",
            "videomae_prediction",
            "videomae_confidence",
            "child_detection_rate",
            "object_classes",
            "expected_hazard",
            "predicted_hazard",
            "valid_json",
            "correct",
            "reason",
            "end_to_end_seconds",
        ]
    ]
)

display(system_tests_df)

# ------------------------------------------------------------
# 10. Sauvegardes
# ------------------------------------------------------------

E2E_RESULTS_PATH = PROJECT_DIR / "e2e_final_results.csv"
E2E_TESTS_PATH = PROJECT_DIR / "e2e_system_tests.csv"
E2E_SUMMARY_PATH = PROJECT_DIR / "e2e_final_summary.json"

e2e_results_df.to_csv(E2E_RESULTS_PATH, index=False)
system_tests_df.to_csv(E2E_TESTS_PATH, index=False)

summary = {
    "pipeline": (
        "raw video -> frames -> YOLO26m -> VideoMAE -> YOLOE "
        "-> child-object geometry -> factual JSON -> Mistral few-shot"
    ),
    "final_model": FINAL_MODEL_ID,
    "decision_mode": "few-shot frozen",
    "number_of_videos": int(len(e2e_results_df)),
    "number_correct": int(e2e_results_df["correct"].sum()),
    "sample_accuracy": accuracy,
    "valid_json_rate": valid_json_rate,
    "mean_end_to_end_latency_seconds": mean_e2e_latency,
    "system_tests_passed": int(system_tests_df["passed"].sum()),
    "system_tests_total": int(len(system_tests_df)),
}

with E2E_SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nFichiers sauvegardés :")
print(" -", E2E_RESULTS_PATH)
print(" -", E2E_TESTS_PATH)
print(" -", E2E_SUMMARY_PATH)
print("\nTEST END-TO-END TERMINÉ")


## Interprétation pour le rapport

Ce notebook constitue un **test système**, et non une nouvelle phase de sélection de modèle.

Les hyperparamètres, modèles et prompts sont gelés avant le test. Les cinq vidéos servent uniquement à vérifier que les interfaces entre modules fonctionnent réellement depuis la vidéo brute jusqu'à la décision finale.

À reporter :

- nombre de tests système `PASS/TOTAL` ;
- taux de JSON valides ;
- nombre de décisions correctes sur les 5 cas ;
- latence moyenne end-to-end ;
- au moins un exemple correct et, s'il existe, un exemple d'erreur pour l'analyse qualitative.

Le score sur seulement 5 vidéos est un **test d'intégration système illustratif** et ne remplace pas les métriques finales obtenues sur les 24 vidéos de test de la Partie 4.
